# Fine-tuning Stable Diffusion 2.1 con LoRA (alternativa leggera al fine-tuning completo)

Questo notebook affianca `02_SD21_Filtered_100steps.ipynb` con una variante che addestra la U-Net di Stable Diffusion 2.1 usando **LoRA** (Low-Rank Adaptation, Hu et al. 2021) invece del fine-tuning completo: solo piccole matrici a basso rango vengono aggiornate (i pesi originali restano congelati), riducendo drasticamente i parametri allenabili, la VRAM richiesta e, potenzialmente, il costo energetico del training.

Domanda alla base del confronto: **a parità di dati, step di training e protocollo di selezione checkpoint, LoRA raggiunge una qualità generativa comparabile al fine-tuning completo di `02`, con un costo computazionale nettamente inferiore?** È un'estensione della Domanda di ricerca D1 (qualità della generazione) con una lente di efficienza/sostenibilità (D3).

Per isolare l'effetto del **metodo** di fine-tuning (LoRA vs completo) dagli altri fattori, questo notebook riusa:

- **stessi dati** di `02` (`data/processed/` + `data/real_augmented/`);
- **stesso modello base** SD2.1 originale (VAE e text encoder congelati, come in `02`; nessun VAE adattato come in `03`);
- **stessi step totali, stessa frequenza di checkpoint, stesso batch effettivo** di `02` (8000 step, checkpoint ogni 500, batch 2 × grad-accum 4);
- **stessa pipeline di valutazione/generazione/filtro/test**, riutilizzata dinamicamente da `02` (sezioni 5-10), con l'unica differenza tecnica che i checkpoint LoRA vengono caricati come adapter sopra la pipeline base invece che come U-Net completa (vedi sezione 5).

Cosa cambia rispetto a `02`: script di training (`train_text_to_image_lora.py` invece di `train_text_to_image.py`), rango LoRA, learning rate più alto (prassi standard per LoRA) e assenza di 8-bit Adam (non supportato dallo script LoRA, e comunque meno necessario con così pochi parametri allenabili).

**Cartelle dedicate:**

- esperimento: `experiments/diffusers/04_sd21_lora/`
- risultati: `results/2_diffusers/04_sd21_lora/`
- immagini sintetiche finali filtrate: `data/synthetic/04_sd21_lora/{positive, negative}/`


## 1. Ambiente e configurazione

Le celle seguenti impostano l'ambiente di esecuzione, individuano la root del progetto, verificano PyTorch/GPU e definiscono una **singola configurazione condivisa** dall'intero notebook.

Rispetto a `02`:

- `EXPERIMENT_NAME` e `RESULTS_04_DIR` sono nuovi e isolano gli artefatti da quelli precedenti;
- viene aggiunta la dipendenza `peft` (richiesta da `train_text_to_image_lora.py` per gli adapter LoRA);
- vengono aggiunti `LORA_RANK` e un `LEARNING_RATE` più alto, dedicati al training LoRA;
- `MAX_TRAIN_STEPS`, `CHECKPOINTING_STEPS`, `CHECKPOINTS_TOTAL_LIMIT`, batch e gradient accumulation restano **identici a `02`** per un confronto controllato.


In [ ]:
# === Bootstrap unificato notebooks/ ===
# Funziona dalla root del progetto e da ogni sottocartella della struttura notebooks/.
import sys as _sys
from pathlib import Path as _Path


def _find_mammo_root():
    for _candidate in [_Path.cwd().resolve(), *_Path.cwd().resolve().parents]:
        if _candidate.name == "MammoDiffusion":
            return _candidate
        if (_candidate / "data").is_dir() and (_candidate / "notebooks").is_dir():
            return _candidate
    raise FileNotFoundError("Root MammoDiffusion non trovata da " + str(_Path.cwd()))


PROJECT_ROOT = _find_mammo_root()
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
UTILITY_DIR = NOTEBOOKS_DIR / "utility"
for _path in (str(UTILITY_DIR), str(NOTEBOOKS_DIR)):
    if _path not in _sys.path:
        _sys.path.insert(0, _path)

BASE = PROJECT_ROOT
BASE_DIR = PROJECT_ROOT
BASE_PATH = str(PROJECT_ROOT) + "/"
# === Fine bootstrap unificato ===

# Bootstrap dipendenze e import (equivalente 02, EXPERIMENT_NAME aggiornato per 04 + peft per LoRA)
from pathlib import Path
from tempfile import TemporaryDirectory
from datetime import datetime
from contextlib import contextmanager
import gc
import hashlib
import importlib
import importlib.util
import json
import math
import os
import re
import shutil
import subprocess
import sys
import warnings
import zipfile

PROJECT_NAME = "MammoDiffusion"
PROJECT_ROOT_OVERRIDE = None  # Esempio Colab: "/content/drive/MyDrive/MammoDiffusion"
EXPERIMENT_NAME = "diffusers/04_sd21_lora"
DIFFUSERS_REVISION = "3759fab56d3170a04d747e918a13e55fda6681e2"


def find_project_root(project_name=PROJECT_NAME, override=PROJECT_ROOT_OVERRIDE):
    if override is not None:
        root = Path(override).expanduser().resolve()
        if not root.is_dir():
            raise FileNotFoundError(f"PROJECT_ROOT_OVERRIDE non esiste: {root}")
        return root

    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if candidate.name == project_name or (
            (candidate / "data").is_dir() and (candidate / "notebooks").is_dir()
        ):
            return candidate

    fallback_candidates = [
        cwd / project_name,
        Path("/content") / project_name,
        Path("/content/drive/MyDrive") / project_name,
        Path.home() / project_name,
        Path.home() / "Progetto" / project_name,
    ]
    for candidate in fallback_candidates:
        if candidate.is_dir():
            return candidate.resolve()

    raise FileNotFoundError(
        "Root di MammoDiffusion non trovata. Esegui il notebook dalla repository "
        "oppure imposta PROJECT_ROOT_OVERRIDE."
    )


PROJECT_ROOT = find_project_root()
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

EXPERIMENTS_DIR = PROJECT_ROOT / "experiments"
EXPERIMENT_DIR = EXPERIMENTS_DIR / EXPERIMENT_NAME
from shared_diffusers_assets import (DIFFUSERS_REVISION, SHARED_DIFFUSERS_REPO_DIR, SHARED_SD21_BASE_DIR, ensure_shared_diffusers_repo, ensure_diffusers_editable_install, shared_diffusers_train_script, verify_diffusers_revision, verify_shared_sd21_base, shared_sd21_signature)
DIFFUSERS_REPO_DIR = SHARED_DIFFUSERS_REPO_DIR
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

AUTO_INSTALL_PACKAGES = {
    "gdown": "gdown",
    "matplotlib": "matplotlib",
    "numpy": "numpy",
    "pandas": "pandas",
    "PIL": "Pillow",
    "tqdm": "tqdm",
    "IPython": "ipython",
    "transformers": "transformers",
    "accelerate": "accelerate",
    "datasets": "datasets",
    "safetensors": "safetensors",
    "huggingface_hub": "huggingface_hub",
    "bitsandbytes": "bitsandbytes",
    "prdc": "prdc",
    "tensorboard": "tensorboard",
    "peft": "peft",  # richiesto da train_text_to_image_lora.py per gli adapter LoRA
}
missing_packages = [
    package_name
    for module_name, package_name in AUTO_INSTALL_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]
if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])

missing_required = [
    module_name for module_name in ["torch"] if importlib.util.find_spec(module_name) is None
]
if missing_required:
    raise ImportError(
        "Dipendenze richieste non trovate: "
        + ", ".join(missing_required)
        + ". Installa PyTorch nell'ambiente prima di eseguire il notebook."
    )

DIFFUSERS_REPO_DIR = ensure_shared_diffusers_repo()
verify_diffusers_revision(DIFFUSERS_REPO_DIR)
ensure_diffusers_editable_install(DIFFUSERS_REPO_DIR)
TRAIN_SCRIPT = shared_diffusers_train_script(lora=True)
DIFFUSERS_SRC_DIR = DIFFUSERS_REPO_DIR / "src"
importlib.invalidate_caches()
# Ricarica la utility: un kernel gia' attivo potrebbe avere in cache una
# firma precedente di prepare_sd_manifest con il terzo argomento obbligatorio.
import parallel_generation_utils as _parallel_generation_utils
_parallel_generation_utils = importlib.reload(_parallel_generation_utils)

for name in [
    "HF_HUB_VERBOSITY",
    "TRANSFORMERS_VERBOSITY",
    "DIFFUSERS_VERBOSITY",
    "ACCELERATE_LOG_LEVEL",
]:
    os.environ[name] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONWARNINGS"] = "ignore"

import gdown
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from diffusers import StableDiffusionPipeline, UNet2DConditionModel
from generative_evaluator import GenerativeEvaluator
from eco_tracker import measure_sustainability
from IPython.display import display
from matplotlib.patches import Patch
from PIL import Image
from prdc import compute_prdc
from tensorboard.backend.event_processing import event_accumulator
from tqdm.auto import tqdm

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA disponibile:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "nessuna")
print("PROJECT_ROOT:", PROJECT_ROOT)

# Generazione multi-GPU: usata solo da evaluation e dataset finale, mai dal training.
PARALLEL_GENERATION = True
GENERATION_GPU_DEVICES = "auto"
GENERATION_MAX_WORKERS = None
GENERATION_SCHEDULER = "dynamic_reservations"
GENERATION_RESERVATION_SIZE = 4
EVALUATION_GENERATION_SCHEDULER = "auto"
SD_CHECKPOINT_TYPE = "lora"

def _sd_parallel_generate_checkpoint_jobs(checkpoints, base_model_dir, eval_dir, negative_prompt, positive_prompt, n_gen, inference_steps, guidance_scale, seed, resolution):
    from parallel_generation_utils import SD_SEED_OFFSETS, missing_named_png_indices, prepare_sd_manifest, run_sd_generation_jobs
    jobs = []
    for step, checkpoint_path in checkpoints:
        requests = []
        for class_name, prompt in (("negative", negative_prompt), ("positive", positive_prompt)):
            class_offset = SD_SEED_OFFSETS[f"evaluation:{class_name}"]
            out_dir = Path(eval_dir) / Path(checkpoint_path).name / class_name
            request = {"name": class_name, "class_name": class_name, "phase": "evaluation", "prompt": prompt, "out_dir": str(out_dir), "count": n_gen, "seed": int(seed), "class_offset": class_offset, "inference_steps": int(inference_steps), "guidance_scale": float(guidance_scale), "resolution": int(resolution), "checkpoint_type": SD_CHECKPOINT_TYPE, "base_model_dir": str(Path(base_model_dir).resolve())}
            prepare_sd_manifest(request, str(checkpoint_path))
            missing = missing_named_png_indices(out_dir, n_gen)
            if missing:
                requests.append({**request, "indices": missing})
        if requests:
            jobs.append({"label": f"checkpoint {step}", "checkpoint_path": str(checkpoint_path), "checkpoint_type": SD_CHECKPOINT_TYPE, "base_model_dir": str(base_model_dir), "requests": requests})
    if not jobs:
        print("Stable Diffusion evaluation: nessuna immagine mancante o corrotta.")
        return []
    return run_sd_generation_jobs(jobs, GENERATION_GPU_DEVICES if PARALLEL_GENERATION else "off", GENERATION_MAX_WORKERS, Path(EXPERIMENT_DIR) / "logs" / "parallel_generation", Path(PROJECT_ROOT), dry_run=False, generation_scheduler=EVALUATION_GENERATION_SCHEDULER, reservation_size=GENERATION_RESERVATION_SIZE)


def _sd_parallel_generate_final(checkpoint_path, base_model_dir, prompt, out_dir, target_total, inference_steps, guidance_scale, seed, resolution, class_name, reused_prefix, n_reused):
    from parallel_generation_utils import SD_SEED_OFFSETS, final_sd_generation_plan, prepare_sd_manifest, run_sd_final_generation
    n_new = int(target_total) - int(n_reused)
    if n_new < 0:
        raise RuntimeError(f"Immagini evaluation riusate ({n_reused}) oltre il target ({target_total}).")
    request = {"name": class_name, "class_name": class_name, "phase": "final_new", "prompt": prompt, "out_dir": str(out_dir), "indices": [], "count": n_new, "seed": int(seed), "class_offset": SD_SEED_OFFSETS[f"final_new:{class_name}"], "inference_steps": int(inference_steps), "guidance_scale": float(guidance_scale), "resolution": int(resolution), "checkpoint_type": SD_CHECKPOINT_TYPE, "base_model_dir": str(Path(base_model_dir).resolve())}
    prepare_sd_manifest(request, str(checkpoint_path))
    plan = final_sd_generation_plan(Path(out_dir), target_total, reused_prefix)
    missing = plan["missing_gen_indices"]
    request["indices"] = missing
    if not missing:
        return plan
    run_sd_final_generation(Path(checkpoint_path), SD_CHECKPOINT_TYPE, Path(base_model_dir), [request], GENERATION_GPU_DEVICES if PARALLEL_GENERATION else "off", GENERATION_MAX_WORKERS, Path(EXPERIMENT_DIR) / "logs" / "parallel_generation", Path(PROJECT_ROOT), dry_run=False, generation_scheduler=GENERATION_SCHEDULER, reservation_size=GENERATION_RESERVATION_SIZE)
    return final_sd_generation_plan(Path(out_dir), target_total, reused_prefix)

In [ ]:
# IDEMPOTENT_PHASE_MODES_V1
TRAIN_MODE = "auto"       # auto | run | skip
GENERATION_MODE = "auto"  # auto | run | skip
EVALUATION_MODE = "auto"  # auto | run | skip | recompute
FILTER_MODE = "auto"      # auto | run | skip | recompute
PLAN_ONLY = False
ALLOW_HEAVY_RETRAIN = False       # must be True for auto mode to retrain from scratch
ALLOW_FULL_REGENERATION = False   # must be True for auto mode to regenerate a full image set

from artifact_phase_planner import plan_experiment, print_plan, phase_should_run
PHASE_MODES = {"training": TRAIN_MODE, "generation": GENERATION_MODE,
               "evaluation": EVALUATION_MODE, "filter": FILTER_MODE}
ALLOW_FLAGS = {"training": ALLOW_HEAVY_RETRAIN, "generation": ALLOW_FULL_REGENERATION}
PHASE_PLAN = plan_experiment(EXPERIMENT_DIR, PHASE_MODES, ALLOW_FLAGS)
print_plan(PHASE_PLAN)


In [ ]:
# Dataset condivisi dal progetto e configurazione dedicata a 04
DATA_DIR = PROJECT_ROOT / "data"
ARCHIVES_DIR = DATA_DIR / "archives"
DATA_PROCESSED_DIR = DATA_DIR / "processed"
DATA_AUG = DATA_DIR / "real_augmented"

PROCESSED_DRIVE_ID = "1qQral_BIBlMl0QN3PllJukdYTOmNGWr3"
AUGMENTED_DRIVE_ID = "1XRc0SxLEPP-zbMJApn4ruaiH8u_rDc-0"
PROCESSED_ZIP_PATH = ARCHIVES_DIR / "processed.zip"
AUGMENTED_ZIP_PATH = ARCHIVES_DIR / "real_augmented.zip"

# Esperimento e repository Diffusers definiti nel bootstrap iniziale
SD21_MODEL_DRIVE_ID = "10XRn-bxpp7tP6ROWLYpeCNYJZIaHfMUt"
SHARED_PRETRAINED_ROOT = NOTEBOOKS_DIR / "pretrained_model"
PRETRAINED_MODEL_DIR = SHARED_SD21_BASE_DIR
PRETRAINED_MODEL_ZIP_PATH = SHARED_PRETRAINED_ROOT / "archives" / "stable-diffusion-2-1-base.zip"
FORCE_MODEL_REDOWNLOAD = False

HF_CACHE_DIR = PROJECT_ROOT / ".cache" / "huggingface" / "04_sd21_lora"
SD_OUTPUT_DIR = EXPERIMENT_DIR / "model"

# Prompt condizionati dalle label (identici a 01/02/03 per confrontabilità)
POSITIVE_PROMPT = (
    "grayscale MLO mammogram, breast cancer positive, malignant finding, "
    "suspicious lesion, medical imaging"
)
NEGATIVE_PROMPT = (
    "grayscale MLO mammogram, breast cancer negative, no malignant finding, "
    "normal screening mammogram, medical imaging"
)

# Fine-tuning LoRA: step totali/checkpoint/batch identici a 02, per confronto controllato.
# LEARNING_RATE e LORA_RANK sono invece specifici di LoRA (vedi sezione 4).
RESOLUTION = 512
TRAIN_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 1e-4  # 02/03 (full fine-tune): 1e-5. LoRA ha molti meno parametri
                       # allenabili e converge meglio con un LR di un ordine superiore
                       # (prassi standard, vedi esempio ufficiale Diffusers LoRA).
LORA_RANK = 16         # dimensione delle matrici di aggiornamento a basso rango (default Diffusers: 4)
MAX_TRAIN_STEPS = 8000
CHECKPOINTING_STEPS = 500
CHECKPOINTS_TOTAL_LIMIT = 32
RESUME_FROM_CHECKPOINT = "latest"
TRAIN_SEED = 42

# Valutazione e generazione (identiche a 02, cartelle dedicate)
N_EVAL_IMAGES_PER_CLASS = 100
N_VALIDATION_IMAGES_PER_CLASS = 73
N_TEST_IMAGES_PER_CLASS = 73
INFERENCE_STEPS = 100
EVAL_GUIDANCE_SCALE = 7.5
EVAL_SEED = 42
PRDC_NEAREST_K = 5
N_FINAL_IMAGES_PER_CLASS = 2722
FINAL_GENERATE_CLASSES = ["positive", "negative"]
N_SELECTED_PER_CLASS = 1361
RAW_MATCHED_SEED = 42
RAW_MATCHED_COUNT = N_SELECTED_PER_CLASS
RAW_MATCHED_ROOT = EXPERIMENT_DIR / "generated_images" / "raw_matched_1361"
NONBLACK_THRESHOLD = 10
FORCE_RECOMPUTE_VALIDATION_COMPARISON = False
FORCE_RECOMPUTE_FINAL_TEST = False

VALIDATION_METADATA_PATH = DATA_PROCESSED_DIR / "metadata" / "val.csv"
TEST_METADATA_PATH = DATA_PROCESSED_DIR / "metadata" / "test.csv"
TRAIN_METADATA_PATH = DATA_PROCESSED_DIR / "metadata" / "train.csv"

RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_04_DIR = RESULTS_DIR / "diffusers/04_sd21_lora"
METRICS_DIR = RESULTS_04_DIR / "metrics"
PLOTS_DIR = RESULTS_04_DIR / "plots"
ECOTRACKER_DIR = RESULTS_04_DIR / "ecotracker"

EVAL_DIR = EXPERIMENT_DIR / "eval_checkpoints"
EVAL_METRICS_PATH = METRICS_DIR / "checkpoint_validation_metrics.json"
EVAL_SUSTAINABILITY_LOG = ECOTRACKER_DIR / "sustainability_validation.jsonl"

FINAL_GEN_DIR = EXPERIMENT_DIR / "generated_images" / "final"
FINAL_NEG_DIR = FINAL_GEN_DIR / "negative"
FINAL_POS_DIR = FINAL_GEN_DIR / "positive"
FINAL_DIRS = {"negative": FINAL_NEG_DIR, "positive": FINAL_POS_DIR}
FINAL_CLASS_LABELS = {"negative": 0, "positive": 1}
RAW_MATCHED_DIRS = {
    class_name: RAW_MATCHED_ROOT / class_name
    for class_name in FINAL_GENERATE_CLASSES
}
RAW_MATCHED_MANIFEST_PATHS = {
    class_name: RAW_MATCHED_ROOT / f"{class_name}_manifest.json"
    for class_name in FINAL_GENERATE_CLASSES
}

SYNTHETIC_DIR = DATA_DIR / "synthetic"
SYNTHETIC_LORA_DIR = SYNTHETIC_DIR / "04_sd21_lora"
FILTERED_DIRS = {
    class_name: SYNTHETIC_LORA_DIR / class_name
    for class_name in FINAL_GENERATE_CLASSES
}
FILTER_REPORT_PATHS = {
    class_name: METRICS_DIR / f"filter_report_{class_name}_adaptive_mask.csv"
    for class_name in FINAL_GENERATE_CLASSES
}
FILTER_SUMMARY_PATHS = {
    class_name: METRICS_DIR / f"filter_summary_{class_name}_adaptive_mask.json"
    for class_name in FINAL_GENERATE_CLASSES
}

VALIDATION_COMPARISON_CSV = METRICS_DIR / "validation_comparison_100_steps_raw_matched_vs_filtered.csv"
VALIDATION_COMPARISON_JSON = METRICS_DIR / "validation_comparison_100_steps_raw_matched_vs_filtered.json"
FINAL_TEST_METRICS_PATH = METRICS_DIR / "final_test_metrics.json"
FINAL_TEST_METRICS_CSV = METRICS_DIR / "final_test_metrics.csv"
FINAL_TEST_RAW_MATCHED_METRICS_PATH = METRICS_DIR / "final_test_metrics_raw_matched_100_steps.json"
FINAL_TEST_RAW_MATCHED_METRICS_CSV = METRICS_DIR / "final_test_metrics_raw_matched_100_steps.csv"

SUSTAINABILITY_LOG = ECOTRACKER_DIR / "sustainability_finetuning.jsonl"
FINAL_GENERATION_LOG = ECOTRACKER_DIR / "sustainability_generation.jsonl"
GENERATION_INFO_PATH = METRICS_DIR / "generation_info.json"

for directory in [
    ARCHIVES_DIR,
    DATA_AUG,
    EXPERIMENT_DIR,
    PRETRAINED_MODEL_ZIP_PATH.parent,
    HF_CACHE_DIR,
    SD_OUTPUT_DIR,
    EVAL_DIR,
    METRICS_DIR,
    PLOTS_DIR,
    ECOTRACKER_DIR,
    FINAL_NEG_DIR,
    FINAL_POS_DIR,
    RAW_MATCHED_ROOT,
    *RAW_MATCHED_DIRS.values(),
    SYNTHETIC_LORA_DIR,
    *FILTERED_DIRS.values(),
]:
    directory.mkdir(parents=True, exist_ok=True)


def label_to_prompt(label):
    prompts = {0: NEGATIVE_PROMPT, 1: POSITIVE_PROMPT}
    try:
        return prompts[int(label)]
    except KeyError as exc:
        raise ValueError(f"Label non valida: {label}") from exc


print("Esperimento          :", EXPERIMENT_NAME)
print("Cartella esperimento :", EXPERIMENT_DIR)
print("Cartella risultati 04:", RESULTS_04_DIR)
print("Cartella sintetiche  :", SYNTHETIC_LORA_DIR)
print("LoRA rank            :", LORA_RANK)
print("Learning rate        :", LEARNING_RATE)
print("Inference step       :", INFERENCE_STEPS)

## 2. Preparazione e verifica dei dati

Identica a `02`: verifica/download di `data/processed/` e `data/real_augmented/`, normalizzazione del metadata e `stage_training_dataset` per uno staging temporaneo compatibile con Diffusers. Copiata così com'è da `02` per restare autosufficiente (nessun import di notebook durante il bootstrap), stessa convenzione già usata da `03`.


In [ ]:
# Utility dati (copia autosufficiente da 02, evita import ipynb durante il bootstrap)
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
EXPECTED_SPLITS = ("train", "val", "test")
EXPECTED_LABELS = ("0", "1")


def count_images(directory):
    directory = Path(directory)
    return sum(
        path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
        for path in directory.rglob("*")
    ) if directory.is_dir() else 0


def download_zip(drive_id, destination, force=False):
    destination = Path(destination)
    if destination.exists() and not force and zipfile.is_zipfile(destination):
        print("Archivio già presente:", destination)
        return
    destination.unlink(missing_ok=True)
    gdown.download(id=drive_id, output=str(destination), quiet=False)
    if not destination.exists() or not zipfile.is_zipfile(destination):
        raise RuntimeError(f"Download non valido: {destination}")


def processed_dataset_ready(directory):
    directory = Path(directory)
    return all(
        count_images(directory / split / label) > 0
        for split in EXPECTED_SPLITS
        for label in EXPECTED_LABELS
    )


def find_directory(root, predicate, description):
    root = Path(root)
    for candidate in [root, *(path for path in root.rglob("*") if path.is_dir())]:
        if predicate(candidate):
            return candidate
    raise FileNotFoundError(f"Directory {description} non trovata dentro {root}")


def prepare_processed_dataset():
    if processed_dataset_ready(DATA_PROCESSED_DIR):
        print("Dataset processed già pronto.")
        return

    download_zip(PROCESSED_DRIVE_ID, PROCESSED_ZIP_PATH)
    with TemporaryDirectory(prefix="mammo_processed_extract_") as tmp:
        with zipfile.ZipFile(PROCESSED_ZIP_PATH) as archive:
            archive.extractall(tmp)
        source_dir = find_directory(tmp, processed_dataset_ready, "processed")
        shutil.copytree(source_dir, DATA_PROCESSED_DIR, dirs_exist_ok=True)

    if not processed_dataset_ready(DATA_PROCESSED_DIR):
        raise FileNotFoundError("Dataset processed incompleto dopo l'estrazione.")


def augmented_dataset_ready():
    return (DATA_AUG / "metadata.csv").is_file() and count_images(DATA_AUG) > 0


def prepare_augmented_dataset():
    if augmented_dataset_ready():
        print("Dataset augmented già pronto.")
        return

    download_zip(AUGMENTED_DRIVE_ID, AUGMENTED_ZIP_PATH)
    with TemporaryDirectory(prefix="mammo_augmented_extract_") as tmp:
        with zipfile.ZipFile(AUGMENTED_ZIP_PATH) as archive:
            archive.extractall(tmp)
        source_dir = find_directory(
            tmp,
            lambda path: (path / "metadata.csv").is_file() and count_images(path) > 0,
            "augmented con metadata.csv",
        )
        DATA_AUG.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source_dir / "metadata.csv", DATA_AUG / "metadata.csv")
        for image_path in source_dir.rglob("*"):
            if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS:
                destination = DATA_AUG / image_path.name
                if not destination.exists():
                    shutil.copy2(image_path, destination)

    if not augmented_dataset_ready():
        raise FileNotFoundError("Dataset augmented incompleto dopo l'estrazione.")


def load_training_metadata(data_aug):
    metadata_path = Path(data_aug) / "metadata.csv"
    metadata = pd.read_csv(metadata_path).copy()
    required = {"file_name", "label"}
    missing = required.difference(metadata.columns)
    if missing:
        raise ValueError(f"Colonne mancanti in {metadata_path}: {sorted(missing)}")
    metadata["file_name"] = metadata["file_name"].astype(str).str.replace("\\", "/", regex=False)
    metadata["label"] = metadata["label"].astype(int)
    metadata["text"] = metadata["label"].map(label_to_prompt)
    return metadata


def is_valid_training_image(path):
    path = Path(path)
    return (
        path.is_file()
        and not path.is_symlink()
        and path.suffix.lower() in IMAGE_EXTENSIONS
    )


def resolve_training_image_path(row):
    file_name = Path(str(row["file_name"]))
    label = str(int(row["label"]))
    source = str(row.get("source", "")).strip().lower()
    real_candidate = DATA_PROCESSED_DIR / "train" / label / file_name.name
    augmented_candidate = DATA_AUG / file_name.name

    if source == "real":
        candidates = [real_candidate]
    elif source in {"positive_augmentation", "augmentation", "augmented"}:
        candidates = [augmented_candidate]
    else:
        candidates = [augmented_candidate, real_candidate, PROJECT_ROOT / file_name]

    original_value = row.get("original_processed_path")
    if source == "real" and pd.notna(original_value) and str(original_value).strip():
        original = Path(str(original_value))
        if "data" in original.parts:
            candidates.append(PROJECT_ROOT / Path(*original.parts[original.parts.index("data"):]))
        candidates.append(original)

    for candidate in candidates:
        if is_valid_training_image(candidate):
            return candidate.resolve()

    checked = ", ".join(str(path) for path in candidates)
    raise FileNotFoundError(
        f"Immagine valida non trovata per file_name={row['file_name']}, source={source}. "
        f"Percorsi controllati: {checked}"
    )


def stage_training_dataset(metadata_df, staging_dir):
    """Copia ogni campione in uno staging temporaneo compatibile con HF imagefolder."""
    staging_dir = Path(staging_dir)
    staged = metadata_df.copy()
    staged_names = []

    for index, (_, row) in enumerate(staged.iterrows()):
        source_path = resolve_training_image_path(row)
        destination_name = f"image_{index:06d}{source_path.suffix.lower()}"
        shutil.copy2(source_path, staging_dir / destination_name)
        staged_names.append(destination_name)

    staged["file_name"] = staged_names
    staged.to_csv(staging_dir / "metadata.csv", index=False)

    if count_images(staging_dir) != len(staged):
        raise RuntimeError("Lo staging temporaneo non contiene tutti i campioni attesi.")
    return staged

In [ ]:
prepare_processed_dataset()
prepare_augmented_dataset()
metadata_df = load_training_metadata(DATA_AUG)

print("Campioni training:", len(metadata_df))
print("\nDistribuzione label:")
print(metadata_df["label"].value_counts().sort_index())
if "source" in metadata_df.columns:
    print("\nDistribuzione source:")
    print(metadata_df["source"].value_counts())

sample_df = metadata_df.sample(min(6, len(metadata_df)), random_state=42)
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for axis, (_, row) in zip(axes.flat, sample_df.iterrows()):
    image_path = resolve_training_image_path(row)
    with Image.open(image_path) as image:
        axis.imshow(image.convert("L"), cmap="gray")
    axis.set_title(f'label={row["label"]} | source={row.get("source", "n/a")}')
    axis.axis("off")
for axis in axes.flat[len(sample_df):]:
    axis.axis("off")
plt.tight_layout()
plt.show()

## 3. Preparazione del modello base

Identica a `02`: scarica/verifica il modello Stable Diffusion 2.1 originale (VAE e text encoder non toccati). A differenza di `03`, qui **non** viene costruita alcuna copia con VAE adattato: LoRA viene confrontato con `02` a parità di modello base, isolando l'effetto del solo metodo di fine-tuning della U-Net.


In [ ]:
MODEL_WEIGHT_ALIASES = {
    "text_encoder": ("model.fp16.safetensors", "model.safetensors"),
    "unet": ("diffusion_pytorch_model.fp16.safetensors", "diffusion_pytorch_model.safetensors"),
    "vae": ("diffusion_pytorch_model.fp16.safetensors", "diffusion_pytorch_model.safetensors"),
}


def has_diffusers_structure(model_dir):
    model_dir = Path(model_dir)
    required = ("model_index.json", "scheduler", "tokenizer", "text_encoder", "vae", "unet")
    return all((model_dir / item).exists() for item in required)


def local_sd_model_ready(model_dir):
    model_dir = Path(model_dir)
    weights = (
        model_dir / "text_encoder" / "model.safetensors",
        model_dir / "unet" / "diffusion_pytorch_model.safetensors",
        model_dir / "vae" / "diffusion_pytorch_model.safetensors",
    )
    return has_diffusers_structure(model_dir) and all(path.is_file() for path in weights)


def create_model_weight_copies(model_dir):
    """Crea sempre copie fisiche con i nomi standard attesi da Diffusers."""
    model_dir = Path(model_dir)
    for subfolder, (source_name, target_name) in MODEL_WEIGHT_ALIASES.items():
        source_path = model_dir / subfolder / source_name
        target_path = model_dir / subfolder / target_name
        if target_path.is_symlink():
            target_path.unlink()
        elif target_path.exists():
            continue
        if not source_path.is_file():
            raise FileNotFoundError(f"Peso sorgente non trovato: {source_path}")
        shutil.copy2(source_path, target_path)


def prepare_pretrained_model():
    if local_sd_model_ready(PRETRAINED_MODEL_DIR) and not FORCE_MODEL_REDOWNLOAD:
        print("Modello Stable Diffusion 2.1 già pronto.")
        return PRETRAINED_MODEL_DIR.resolve()

    download_zip(SD21_MODEL_DRIVE_ID, PRETRAINED_MODEL_ZIP_PATH, FORCE_MODEL_REDOWNLOAD)
    with TemporaryDirectory(prefix="mammo_sd21_extract_") as tmp:
        with zipfile.ZipFile(PRETRAINED_MODEL_ZIP_PATH) as archive:
            archive.extractall(tmp)
        source_dir = find_directory(tmp, has_diffusers_structure, "modello Diffusers")
        if PRETRAINED_MODEL_DIR.exists():
            shutil.rmtree(PRETRAINED_MODEL_DIR)
        shutil.copytree(source_dir, PRETRAINED_MODEL_DIR)
        create_model_weight_copies(PRETRAINED_MODEL_DIR)

    if not local_sd_model_ready(PRETRAINED_MODEL_DIR):
        raise FileNotFoundError("Modello Stable Diffusion 2.1 incompleto dopo l'estrazione.")
    return PRETRAINED_MODEL_DIR.resolve()


LOCAL_MODEL_DIR = prepare_pretrained_model()
print("Modello locale:", LOCAL_MODEL_DIR)

In [ ]:
# Verifica del repository Diffusers locale e dello script di training LoRA.
if not TRAIN_SCRIPT.is_file():
    raise FileNotFoundError(f"Script di training non trovato: {TRAIN_SCRIPT}")

print("Diffusers revision:", DIFFUSERS_REVISION)
print("Script training LoRA:", TRAIN_SCRIPT)

## 4. Fine-tuning LoRA della U-Net

Usa `train_text_to_image_lora.py` (stesso repository/revisione Diffusers di `02`/`03`) invece di `train_text_to_image.py`. Differenze rispetto al comando di `02`:

- solo le matrici LoRA (rango `LORA_RANK`) sono allenabili, iniettate nei layer di attenzione della U-Net; i pesi originali restano congelati;
- `--learning_rate` più alto (`LEARNING_RATE`) rispetto a `02`/`03`: con così pochi parametri allenabili, LoRA richiede tipicamente un learning rate di un ordine di grandezza superiore per convergere in un numero di step comparabile (prassi standard, vedi esempio ufficiale Diffusers);
- nessun `--use_8bit_adam`: lo script LoRA non lo supporta, ed è comunque meno necessario dato il numero ridotto di stati dell'ottimizzatore;
- `--validation_prompt` (singolare) invece di `--validation_prompts`: limite dello script LoRA di Diffusers, che accetta un solo prompt di validazione invece di uno per classe.

Step totali, frequenza di checkpoint, batch size e gradient accumulation restano identici a `02` per un confronto controllato.


In [ ]:
def build_training_command(train_data_dir):
    command = [
        sys.executable, "-m", "accelerate.commands.launch",
        "--mixed_precision=bf16",  # fp16 causerebbe "Attempting to unscale FP16 gradients" con LoRA + gradient_checkpointing (i grad ricomputati nel backward restano fp16 anche dopo cast_training_params dello script). bf16 ha lo stesso footprint di memoria di fp16, range dinamico di fp32, non richiede GradScaler.
        "--num_processes=1",
        str(TRAIN_SCRIPT),
        "--pretrained_model_name_or_path", str(LOCAL_MODEL_DIR),
        "--train_data_dir", str(train_data_dir),
        "--image_column", "image",
        "--caption_column", "text",
        "--resolution", str(RESOLUTION),
        "--center_crop",
        "--train_batch_size", str(TRAIN_BATCH_SIZE),
        "--gradient_accumulation_steps", str(GRADIENT_ACCUMULATION_STEPS),
        "--gradient_checkpointing",
        "--max_train_steps", str(MAX_TRAIN_STEPS),
        "--learning_rate", str(LEARNING_RATE),
        "--lr_scheduler", "constant",
        "--lr_warmup_steps", "0",
        "--max_grad_norm", "1",
        "--rank", str(LORA_RANK),
        "--checkpointing_steps", str(CHECKPOINTING_STEPS),
        "--checkpoints_total_limit", str(CHECKPOINTS_TOTAL_LIMIT),
        "--validation_prompt", POSITIVE_PROMPT,
        "--num_validation_images", "2",
        "--validation_epochs", "2",
        "--seed", str(TRAIN_SEED),
        "--cache_dir", str(HF_CACHE_DIR),
        "--output_dir", str(SD_OUTPUT_DIR),
        "--report_to", "tensorboard",
        "--logging_dir", str(SD_OUTPUT_DIR / "logs"),
        "--dataloader_num_workers", "0",
    ]
    if RESUME_FROM_CHECKPOINT is not None:
        command.extend(["--resume_from_checkpoint", RESUME_FROM_CHECKPOINT])
    return command


def build_training_environment():
    environment = os.environ.copy()
    environment["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    conda_prefix = Path(sys.prefix)
    cuda_lib_paths = [
        conda_prefix / "lib",
        conda_prefix / "targets" / "x86_64-linux" / "lib",
        conda_prefix / "lib" / "python3.11" / "site-packages" / "nvidia" / "cu13" / "lib",
        conda_prefix / "lib" / "python3.11" / "site-packages" / "nvidia" / "nvjitlink" / "lib",
    ]
    existing = environment.get("LD_LIBRARY_PATH", "")
    environment["LD_LIBRARY_PATH"] = ":".join(
        [str(path) for path in cuda_lib_paths if path.exists()] + [existing]
    )
    return environment


run_label = f"finetune_sd21_lora_rank{LORA_RANK}_to_{MAX_TRAIN_STEPS}_steps"
if RESUME_FROM_CHECKPOINT is not None:
    run_label += f"_resume_{RESUME_FROM_CHECKPOINT}"
temporary_training = TemporaryDirectory(prefix="mammo_sd21_lora_train_copy_")
try:
    temporary_train_dir = Path(temporary_training.name)
    staged_metadata = stage_training_dataset(metadata_df, temporary_train_dir)
    command = build_training_command(temporary_train_dir)

    print("Staging temporaneo:", temporary_train_dir)
    print("Campioni copiati:", len(staged_metadata))
    print("Comando fine-tuning LoRA:\n", " ".join(map(str, command)))

    with measure_sustainability(label=run_label, sample_interval=0.5) as eco:
        process = subprocess.Popen(
            command,
            env=build_training_environment(),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in iter(process.stdout.readline, ""):
            print(line, end="", flush=True)
        process.wait()
finally:
    temporary_training.cleanup()
    print("Staging temporaneo eliminato.")

record = eco.metrics.to_dict()
record.update({
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "max_train_steps": MAX_TRAIN_STEPS,
    "resume_from_checkpoint": RESUME_FROM_CHECKPOINT,
    "resolution": RESOLUTION,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "learning_rate": LEARNING_RATE,
    "lora_rank": LORA_RANK,
    "source_metadata": str(DATA_AUG / "metadata.csv"),
    "staging_strategy": "temporary_copy",
    "output_dir": str(SD_OUTPUT_DIR),
    "returncode": process.returncode,
    "status": "completed" if process.returncode == 0 else "failed",
})
with SUSTAINABILITY_LOG.open("a", encoding="utf-8") as handle:
    handle.write(json.dumps(record, ensure_ascii=False) + "\n")

print("Metriche sostenibilità:", eco.metrics)
if process.returncode != 0:
    raise subprocess.CalledProcessError(process.returncode, command)

### Curva della loss di training


In [ ]:
event_files = sorted((SD_OUTPUT_DIR / "logs").rglob("events.out.tfevents.*"))
loss_rows = []

for event_file in event_files:
    try:
        accumulator = event_accumulator.EventAccumulator(
            str(event_file),
            size_guidance={"scalars": 0},
        )
        accumulator.Reload()
        scalar_tags = accumulator.Tags().get("scalars", [])
        preferred_tags = ["train_loss", "loss"]
        loss_tag = next((tag for tag in preferred_tags if tag in scalar_tags), None)
        if loss_tag is None:
            loss_tag = next((tag for tag in scalar_tags if tag.endswith("/loss")), None)
        if loss_tag is None:
            continue
        loss_rows.extend({
            "step": event.step,
            "loss": event.value,
            "wall_time": event.wall_time,
            "source": event_file.name,
            "tag": loss_tag,
        } for event in accumulator.Scalars(loss_tag))
    except Exception as exc:
        print(f"Evento TensorBoard non leggibile ({event_file.name}): {exc}")

if not loss_rows:
    print("Nessuna serie di loss TensorBoard disponibile: grafico non generato.")
else:
    df_loss = (
        pd.DataFrame(loss_rows)
        .sort_values(["step", "wall_time"])
        .drop_duplicates(subset="step", keep="last")
    )
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(df_loss["step"], df_loss["loss"], linewidth=1.2)
    ax.set(
        title="Loss di training LoRA registrata da TensorBoard",
        xlabel="Training step",
        ylabel="Loss",
    )
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / "train_loss.png", dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("Curva della loss salvata in:", PLOTS_DIR / "train_loss.png")

## 5. Valutazione dei checkpoint, generazione, filtro e test (codice esplicitato)

Come `03`, questa sezione **importa dinamicamente le celle di codice del notebook `02`** dalla sezione "## 5." in poi (selezione checkpoint, generazione RAW, dataset RAW matched, filtro adattivo, confronto validation, test finale, confronto 50 vs 100 step), per non duplicare centinaia di righe già testate.

**Unica differenza:** un checkpoint LoRA non è una pipeline completa come nel fine-tuning full — contiene solo l'adapter (`pytorch_lora_weights.safetensors`) da applicare sopra la pipeline base. Le funzioni `discover_checkpoints` e `load_pipeline_from_checkpoint` importate da `02` vengono perciò **sovrascritte subito dopo l'import** con una versione compatibile con LoRA; tutte le altre funzioni riutilizzate (generazione, metriche FID/IS/PRDC, filtro, ecc.) restano invariate perché non dipendono dal formato del checkpoint.


Le celle seguenti sono incluse direttamente nel notebook. Il loader LoRA viene ridefinito tra le utility comuni e la valutazione, cosi i checkpoint adapter sono caricati correttamente senza dipendere dal notebook 02.


### 5.1 Valutazione dei checkpoint e selezione sul validation set

Le celle successive valutano tutti i checkpoint salvati, senza utilizzare il test set per prendere decisioni sul modello.

La sezione:

- configura pipeline, metadata di train/validation/test e directory degli output;
- costruisce riferimenti reali temporanei tramite copie fisiche, ricostruendo i percorsi anche quando nei CSV è rimasta una vecchia root assoluta;
- scopre e ordina i checkpoint `checkpoint-<step>` contenenti un UNet valido;
- genera 100 immagini per classe e checkpoint usando **100 inference step**;
- salta la generazione delle immagini già presenti e libera sempre la VRAM con `try/finally`;
- calcola FID, Inception Score e PRDC separatamente per negative e positive, più le medie;
- riusa `results/2_diffusers/02_sd21_filtered_100steps/metrics/checkpoint_validation_metrics.json` quando contiene già tutti i checkpoint, purché contenga anche tutte le metriche PRDC,  evitando ricalcoli non necessari;
- seleziona come `BEST_CHECKPOINT` quello con FID medio più basso sul validation set.

Le metriche PRDC descrivono fedeltà e copertura: precision e density misurano quanto le immagini generate ricadano vicino alla distribuzione reale, mentre recall e coverage misurano quanta parte della variabilità reale venga rappresentata. Le stesse feature
Inception estratte per FID e IS vengono riutilizzate, senza una seconda estrazione. Il parametro `PRDC_NEAREST_K = 5` è fissato nella configurazione ed è condiviso da tutte le valutazioni.

Poiché ciascuna classe dispone di 73 riferimenti reali su validation e test, le stime PRDC possono avere una varianza non trascurabile. Sono particolarmente utili per confronti relativi condotti sullo stesso riferimento; i valori assoluti devono invece essere interpretati con cautela.

Le immagini di valutazione sono conservate in `experiments/.../eval_checkpoints/`; il test set resta riservato esclusivamente alla valutazione finale.

In [ ]:
# Configurazione della selezione checkpoint e della generazione finale
for metadata_path in [VALIDATION_METADATA_PATH, TEST_METADATA_PATH, TRAIN_METADATA_PATH]:
    if not metadata_path.is_file():
        raise FileNotFoundError(f"Metadata split non trovato: {metadata_path}")

print("Checkpoint sorgente       :", SD_OUTPUT_DIR)
print("Validation metadata       :", VALIDATION_METADATA_PATH)
print("Test metadata             :", TEST_METADATA_PATH)
print("Immagini generate/ckpt/cls:", N_EVAL_IMAGES_PER_CLASS)
print("Immagini finali/classe    :", N_FINAL_IMAGES_PER_CLASS)

In [ ]:
# Riferimenti reali temporanei per validation/test, sempre tramite copia
def resolve_split_image_path(raw_path, split_name, label):
    original = Path(str(raw_path)).expanduser()
    candidates = [original] if original.is_absolute() else [PROJECT_ROOT / original]
    if "data" in original.parts:
        candidates.append(PROJECT_ROOT / Path(*original.parts[original.parts.index("data"):]))
    candidates.append(DATA_PROCESSED_DIR / split_name / str(int(label)) / original.name)

    for candidate in dict.fromkeys(candidates):
        if candidate.is_file():
            return candidate.resolve()
    checked = "\n".join(f"  - {candidate}" for candidate in dict.fromkeys(candidates))
    raise FileNotFoundError(f"Immagine non trovata per {raw_path}. Percorsi controllati:\n{checked}")


def get_real_image_paths_from_split_metadata(metadata_path, label, n_images, seed=42):
    metadata_path = Path(metadata_path)
    metadata = pd.read_csv(metadata_path)
    required = {"label", "processed_path"}
    missing = required.difference(metadata.columns)
    if missing:
        raise ValueError(f"Colonne mancanti in {metadata_path}: {sorted(missing)}")

    subset = metadata[metadata["label"].astype(int) == int(label)]
    if subset.empty:
        raise ValueError(f"Nessuna immagine label={label} in {metadata_path}")
    n_take = len(subset) if n_images is None else min(int(n_images), len(subset))
    if n_images is not None and n_take < n_images:
        print(f"Richieste {n_images} immagini label={label}; disponibili {n_take}.")

    subset = subset.sample(n=n_take, random_state=seed)
    return [
        resolve_split_image_path(raw_path, metadata_path.stem.lower(), label)
        for raw_path in subset["processed_path"]
    ]


@contextmanager
def temporary_real_reference_dir_from_split_metadata(metadata_path, label, n_images, seed=42):
    image_paths = get_real_image_paths_from_split_metadata(metadata_path, label, n_images, seed)
    with TemporaryDirectory(prefix=f"real_ref_label{label}_") as tmp:
        tmp_dir = Path(tmp)
        for index, source_path in enumerate(image_paths):
            destination = tmp_dir / f"real_{label}_{index:04d}{source_path.suffix.lower()}"
            shutil.copy2(source_path, destination)
        yield tmp_dir


@contextmanager
def temporary_real_reference_dirs_from_split_metadata(metadata_path, n_per_class, seed=42):
    with temporary_real_reference_dir_from_split_metadata(
        metadata_path, label=0, n_images=n_per_class, seed=seed
    ) as real_neg_dir, temporary_real_reference_dir_from_split_metadata(
        metadata_path, label=1, n_images=n_per_class, seed=seed + 1
    ) as real_pos_dir:
        yield real_neg_dir, real_pos_dir


print("Selezione checkpoint: validation.")
print("Valutazione finale: test, esclusivamente nell'ultima cella.")

In [ ]:
# Utility per checkpoint, generazione e metriche
def discover_checkpoints(output_dir):
    checkpoints = []
    for path in Path(output_dir).glob("checkpoint-*"):
        match = re.fullmatch(r"checkpoint-(\d+)", path.name)
        if match and path.is_dir() and (path / "unet").is_dir():
            checkpoints.append((int(match.group(1)), path))
    return sorted(checkpoints)


def load_pipeline_from_checkpoint(checkpoint_path, base_model_dir, device="cuda"):
    unet = UNet2DConditionModel.from_pretrained(
        str(Path(checkpoint_path) / "unet"),
        torch_dtype=torch.float16,
    )
    pipeline = StableDiffusionPipeline.from_pretrained(
        str(base_model_dir),
        unet=unet,
        torch_dtype=torch.float16,
        safety_checker=None,
        requires_safety_checker=False,
    ).to(device)
    pipeline.set_progress_bar_config(disable=True)
    return pipeline


def count_pngs(directory):
    return sum(path.is_file() and not path.name.startswith(".tmp_") for path in Path(directory).glob("*.png")) if Path(directory).exists() else 0


def generate_images_to_dir(
    pipeline, prompt, out_dir, n, inference_steps, guidance_scale, seed, resolution=512
):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    n_existing = count_pngs(out_dir)
    if n_existing >= n:
        print(f"    {n_existing} immagini già presenti, skip")
        return

    print(f"    {n_existing} già presenti, genero {n - n_existing} mancanti")
    generator = torch.Generator("cuda").manual_seed(seed + n_existing)
    for index in tqdm(range(n_existing, n), desc=f"  Generazione {out_dir.name}", unit="img"):
        image = pipeline(
            prompt,
            num_inference_steps=inference_steps,
            guidance_scale=guidance_scale,
            height=resolution,
            width=resolution,
            generator=generator,
        ).images[0]
        image.save(out_dir / f"gen_{index:04d}.png")


def compute_prdc_metrics(real_features, fake_features, nearest_k=PRDC_NEAREST_K):
    """Calcola PRDC riutilizzando le feature Inception già estratte per FID."""
    metrics = compute_prdc(
        real_features=real_features,
        fake_features=fake_features,
        nearest_k=nearest_k,
    )
    return {
        name: round(float(metrics[name]), 4)
        for name in ("precision", "recall", "density", "coverage")
    }


def eval_one_checkpoint(
    step,
    ckpt_path,
    base_model_dir,
    real_neg_dir,
    real_pos_dir,
    eval_dir,
    n_gen,
    inference_steps,
    guidance_scale,
    seed,
    resolution=512,
):
    class_config = {
        "negative": (NEGATIVE_PROMPT, Path(real_neg_dir), seed),
        "positive": (POSITIVE_PROMPT, Path(real_pos_dir), seed + 1),
    }
    generated_dirs = {
        name: Path(eval_dir) / f"checkpoint-{step}" / name
        for name in class_config
    }

    print(f"\nCheckpoint {step} | {Path(ckpt_path).name}")
    needs_generation = any(count_pngs(path) < n_gen for path in generated_dirs.values())
    pipeline = load_pipeline_from_checkpoint(ckpt_path, base_model_dir) if needs_generation else None
    try:
        for name, (prompt, _, class_seed) in class_config.items():
            if count_pngs(generated_dirs[name]) < n_gen:
                print(f"  Generazione {name}...")
                generate_images_to_dir(
                    pipeline,
                    prompt,
                    generated_dirs[name],
                    n_gen,
                    inference_steps,
                    guidance_scale,
                    class_seed,
                    resolution,
                )
    finally:
        if pipeline is not None:
            del pipeline
            gc.collect()
            torch.cuda.empty_cache()

    metrics = {}
    for name, (_, real_dir, _) in class_config.items():
        print(f"  FID + IS + PRDC: {name}")
        evaluator = GenerativeEvaluator(
            real_dir=real_dir,
            generated_dir=generated_dirs[name],
            batch_size=8,
            num_workers=0,
        )
        fid_is_metrics, real_features, fake_features = evaluator.compute_with_features()
        metrics[name] = {
            **fid_is_metrics,
            **compute_prdc_metrics(real_features, fake_features),
        }

    averages = {
        metric: round(sum(metrics[name][metric] for name in metrics) / len(metrics), 4)
        for metric in ("FID", "IS_mean", "IS_std", "precision", "recall", "density", "coverage")
    }
    print(
        f"  Avg FID={averages['FID']:.4f} | "
        f"IS={averages['IS_mean']:.4f}±{averages['IS_std']:.4f} | "
        f"P={averages['precision']:.4f} | R={averages['recall']:.4f}"
    )
    return {
        "step": step,
        "ckpt_name": Path(ckpt_path).name,
        **metrics,
        "avg_FID": averages["FID"],
        "avg_IS_mean": averages["IS_mean"],
        "avg_IS_std": averages["IS_std"],
        "avg_precision": averages["precision"],
        "avg_recall": averages["recall"],
        "avg_density": averages["density"],
        "avg_coverage": averages["coverage"],
    }


def copy_eval_images_to_final(eval_src_dir, final_dst_dir, max_images, prefix):
    final_dst_dir = Path(final_dst_dir)
    final_dst_dir.mkdir(parents=True, exist_ok=True)
    from parallel_generation_utils import valid_named_png_indices
    eval_images = [
        Path(eval_src_dir) / f"gen_{index:04d}.png"
        for index in valid_named_png_indices(Path(eval_src_dir), max_images)
    ]
    for index, source_path in enumerate(eval_images):
        shutil.copy2(source_path, final_dst_dir / f"{prefix}_{index:04d}.png")
    print(f"  Riutilizzate {len(eval_images)} immagini da {Path(eval_src_dir).name}.")
    return len(eval_images)


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def remove_generated_duplicates_of_reused_images(final_dir, reused_prefix):
    final_dir = Path(final_dir)
    reused_hashes = {file_sha256(path) for path in final_dir.glob(f"{reused_prefix}_*.png")}
    removed = []
    for path in sorted(final_dir.glob("gen_*.png")):
        if file_sha256(path) in reused_hashes:
            path.unlink()
            removed.append(path)
    if removed:
        print(f"  Rimosse {len(removed)} copie gen_* sovrapposte alle immagini riusate.")
    return removed


def duplicate_png_groups(directory):
    by_hash = {}
    for path in sorted(Path(directory).glob("*.png")):
        if path.name.startswith(".tmp_"):
            continue
        by_hash.setdefault(file_sha256(path), []).append(path.name)
    return {digest: names for digest, names in by_hash.items() if len(names) > 1}

In [ ]:
# Loader LoRA locale: i checkpoint contengono adapter, non una U-Net completa.
def discover_checkpoints(output_dir):
    """Come la versione di 02, ma un checkpoint LoRA è valido se contiene
    pytorch_lora_weights.safetensors nella root, non una sottocartella unet/ completa."""
    checkpoints = []
    for path in Path(output_dir).glob("checkpoint-*"):
        match = re.fullmatch(r"checkpoint-(\d+)", path.name)
        if match and path.is_dir() and (path / "pytorch_lora_weights.safetensors").is_file():
            checkpoints.append((int(match.group(1)), path))
    return sorted(checkpoints)


def load_pipeline_from_checkpoint(checkpoint_path, base_model_dir, device="cuda"):
    """Carica la pipeline base (pesi originali SD2.1) e applica sopra gli adapter LoRA
    del checkpoint, invece di sostituire l'intera U-Net come fa la versione full fine-tune."""
    pipeline = StableDiffusionPipeline.from_pretrained(
        str(base_model_dir),
        torch_dtype=torch.float16,
        safety_checker=None,
        requires_safety_checker=False,
    ).to(device)
    pipeline.load_lora_weights(str(checkpoint_path))
    pipeline.set_progress_bar_config(disable=True)
    return pipeline


In [ ]:
# Valutazione di tutti i checkpoint usando esclusivamente il validation
checkpoints = discover_checkpoints(SD_OUTPUT_DIR)
print(f"Trovati {len(checkpoints)} checkpoint:")
for _step, _path in checkpoints:
    print(f"  checkpoint-{_step}")


# La generazione e' completata dai worker prima delle metriche; JSON/cache restano nel processo notebook.
_sd_parallel_generate_checkpoint_jobs(
    checkpoints, LOCAL_MODEL_DIR, EVAL_DIR, NEGATIVE_PROMPT, POSITIVE_PROMPT,
    N_EVAL_IMAGES_PER_CLASS, INFERENCE_STEPS, EVAL_GUIDANCE_SCALE, EVAL_SEED, RESOLUTION,
)

from parallel_generation_utils import checkpoint_content_signature, file_content_signature, png_content_signature, sd_metrics_cache_config, sd_metrics_cache_compatible
CHECKPOINT_EVAL_CACHE_PATH = EVAL_METRICS_PATH.with_name("checkpoint_validation_cache_v2.json")
checkpoint_eval_config = sd_metrics_cache_config(eval_seed=EVAL_SEED, inference_steps=INFERENCE_STEPS, guidance_scale=EVAL_GUIDANCE_SCALE, resolution=RESOLUTION, n_gen_per_class=N_EVAL_IMAGES_PER_CLASS, checkpoint_type=SD_CHECKPOINT_TYPE, base_model_dir=LOCAL_MODEL_DIR, n_validation_images_per_class=N_VALIDATION_IMAGES_PER_CLASS, prdc_nearest_k=PRDC_NEAREST_K, evaluator_batch_size=8, evaluator_num_workers=0, metric_backend="GenerativeEvaluator", metric_backend_version="torchmetrics_fid_is_prdc_v1")
validation_csv_signature = file_content_signature(VALIDATION_METADATA_PATH)
current_checkpoint_inputs = {path.name: {"checkpoint_signature": checkpoint_content_signature(path), "negative_image_signature": png_content_signature(EVAL_DIR / path.name / "negative"), "positive_image_signature": png_content_signature(EVAL_DIR / path.name / "positive")} for _, path in checkpoints}
all_metrics = None
if CHECKPOINT_EVAL_CACHE_PATH.is_file():
    with CHECKPOINT_EVAL_CACHE_PATH.open(encoding="utf-8") as handle:
        cache_payload = json.load(handle)
    cached_entries = cache_payload.get("checkpoints", {}) if sd_metrics_cache_compatible(cache_payload, checkpoint_eval_config, VALIDATION_METADATA_PATH) else {}
    if set(cached_entries) == set(current_checkpoint_inputs) and all(all(cached_entries[name].get(key) == value for key, value in signature.items()) for name, signature in current_checkpoint_inputs.items()):
        all_metrics = [cached_entries[path.name]["metrics"] for _, path in checkpoints]
        print("Metriche validation da cache v2 con firme compatibili.")

if all_metrics is None:
    all_metrics = []
    with temporary_real_reference_dirs_from_split_metadata(
        metadata_path=VALIDATION_METADATA_PATH,
        n_per_class=N_VALIDATION_IMAGES_PER_CLASS,
        seed=EVAL_SEED,
    ) as (real_neg_ref_dir, real_pos_ref_dir):
        print("\nDirectory temporanee validation:")
        print("  Negative:", real_neg_ref_dir)
        print("  Positive:", real_pos_ref_dir)

        with measure_sustainability(label="checkpoint_validation_evaluation", sample_interval=0.5) as eco_eval:
            for step, ckpt_path in checkpoints:
                all_metrics.append(eval_one_checkpoint(
                    step=step,
                    ckpt_path=ckpt_path,
                    base_model_dir=LOCAL_MODEL_DIR,
                    real_neg_dir=real_neg_ref_dir,
                    real_pos_dir=real_pos_ref_dir,
                    eval_dir=EVAL_DIR,
                    n_gen=N_EVAL_IMAGES_PER_CLASS,
                    inference_steps=INFERENCE_STEPS,
                    guidance_scale=EVAL_GUIDANCE_SCALE,
                    seed=EVAL_SEED,
                    resolution=RESOLUTION,
                ))

    with EVAL_METRICS_PATH.open("w", encoding="utf-8") as handle:
        json.dump(all_metrics, handle, indent=2, ensure_ascii=False)
    with CHECKPOINT_EVAL_CACHE_PATH.open("w", encoding="utf-8") as handle:
        json.dump({"schema_version": 2, "config": checkpoint_eval_config, "validation_csv_signature": validation_csv_signature, "checkpoints": {path.name: {**current_checkpoint_inputs[path.name], "metrics": metrics} for (_, path), metrics in zip(checkpoints, all_metrics)}}, handle, indent=2, ensure_ascii=False)

    eco_record = eco_eval.metrics.to_dict()
    eco_record.update({
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "record_type": "checkpoint_validation_evaluation",
        "n_checkpoints": len(checkpoints),
        "n_generated_images_per_class": N_EVAL_IMAGES_PER_CLASS,
        "n_real_images_per_class": N_VALIDATION_IMAGES_PER_CLASS,
        "real_reference_metadata": str(VALIDATION_METADATA_PATH),
    })
    with EVAL_SUSTAINABILITY_LOG.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(eco_record, ensure_ascii=False) + "\n")

    print(f"\nMetriche eco-tracking validation:\n{eco_eval.metrics}")
    print(f"Metriche validation salvate in: {EVAL_METRICS_PATH}")

In [ ]:
# Selezione best checkpoint
# Ricarica da file (idempotente se si ri-esegue la cella)
with open(EVAL_METRICS_PATH, encoding="utf-8") as f:
    all_metrics = json.load(f)

df_eval = pd.DataFrame([
    {
        "checkpoint":    m["ckpt_name"],
        "step":          m["step"],
        "avg_FID":       m["avg_FID"],
        "avg_IS_mean":   m["avg_IS_mean"],
        "avg_IS_std":    m["avg_IS_std"],
        "avg_precision": m["avg_precision"],
        "avg_recall":    m["avg_recall"],
        "avg_density":   m["avg_density"],
        "avg_coverage":  m["avg_coverage"],
        "FID_neg":       m["negative"]["FID"],
        "FID_pos":       m["positive"]["FID"],
        "IS_neg":        m["negative"]["IS_mean"],
        "IS_pos":        m["positive"]["IS_mean"],
        "precision_neg": m["negative"]["precision"],
        "precision_pos": m["positive"]["precision"],
        "recall_neg":    m["negative"]["recall"],
        "recall_pos":    m["positive"]["recall"],
    }
    for m in all_metrics
]).sort_values("avg_FID").reset_index(drop=True)

print("Riepilogo metriche (ordinato per avg_FID crescente - migliore prima):")
print(df_eval.to_string(index=False))

best_row = df_eval.iloc[0]
BEST_CHECKPOINT = SD_OUTPUT_DIR / best_row["checkpoint"]

print(f"\nMiglior checkpoint : {best_row['checkpoint']}")
print(f"  avg FID       : {best_row['avg_FID']:.4f}")
print(f"  avg IS        : {best_row['avg_IS_mean']:.4f} ± {best_row['avg_IS_std']:.4f}")
print(f"  avg precision : {best_row['avg_precision']:.4f}")
print(f"  avg recall    : {best_row['avg_recall']:.4f}")
print(f"  Path          : {BEST_CHECKPOINT}")

# Grafici diagnostici separati; il FID resta nella figura dedicata della cella successiva
df_eval_by_step = df_eval.sort_values("step").reset_index(drop=True)
best_step = int(best_row["step"])

figure, axis = plt.subplots(figsize=(10, 5), constrained_layout=True)
axis.plot(
    df_eval_by_step["step"],
    df_eval_by_step["avg_IS_mean"],
    color="#9467bd",
    marker="o",
    label="IS medio",
)
axis.fill_between(
    df_eval_by_step["step"],
    df_eval_by_step["avg_IS_mean"] - df_eval_by_step["avg_IS_std"],
    df_eval_by_step["avg_IS_mean"] + df_eval_by_step["avg_IS_std"],
    color="#9467bd",
    alpha=0.2,
    label="± deviazione standard",
)
axis.axvline(best_step, color="red", linestyle="--", alpha=0.45, label="best FID")
axis.set(title="Inception Score medio per checkpoint", xlabel="Training step", ylabel="IS")
axis.grid(alpha=0.25)
axis.legend()
figure.savefig(PLOTS_DIR / "is_per_checkpoint.png", dpi=180, bbox_inches="tight")
plt.show()
plt.close(figure)

figure, axis = plt.subplots(figsize=(10, 5), constrained_layout=True)
for column, label, color in [
    ("avg_precision", "Precision media", "#1f77b4"),
    ("avg_recall", "Recall media", "#ff7f0e"),
]:
    axis.plot(
        df_eval_by_step["step"],
        df_eval_by_step[column],
        color=color,
        marker="o",
        label=label,
    )
axis.axvline(best_step, color="red", linestyle="--", alpha=0.45, label="best FID")
axis.set(
    title="Precision e recall medie per checkpoint",
    xlabel="Training step",
    ylabel="Valore PRDC",
)
axis.grid(alpha=0.25)
axis.legend()
figure.savefig(PLOTS_DIR / "precision_recall_over_steps.png", dpi=180, bbox_inches="tight")
plt.show()
plt.close(figure)

figure, axis = plt.subplots(figsize=(10, 5), constrained_layout=True)
for column, label, color in [
    ("avg_density", "Density media", "#2ca02c"),
    ("avg_coverage", "Coverage media", "#d62728"),
]:
    axis.plot(
        df_eval_by_step["step"],
        df_eval_by_step[column],
        color=color,
        marker="o",
        label=label,
    )
axis.axvline(best_step, color="red", linestyle="--", alpha=0.45, label="best FID")
axis.set(
    title="Density e coverage medie per checkpoint",
    xlabel="Training step",
    ylabel="Valore PRDC",
)
axis.grid(alpha=0.25)
axis.legend()
figure.savefig(PLOTS_DIR / "density_coverage_over_steps.png", dpi=180, bbox_inches="tight")
plt.show()
plt.close(figure)

print("Grafici diagnostici checkpoint separati salvati in:", PLOTS_DIR)

#### 5.1a Diagnostica della selezione del checkpoint

I grafici successivi trasformano il JSON di valutazione in una lettura metodologica del percorso di training, senza ricalcolare le metriche. Ogni famiglia di metriche viene mostrata in una figura separata per evitare sovrapposizioni e duplicazioni:

- FID negative, positive e medio in funzione dello step;
- Inception Score medio con banda di deviazione standard;
- precision e recall medie;
- density e coverage medie;
- scatter precision-recall dei checkpoint.

Il checkpoint scelto viene evidenziato, ma la regola di selezione resta esclusivamente il minimo `avg_FID`. Le figure vengono mostrate nel notebook e salvate in `results/2_diffusers/02_sd21_filtered_100steps/plots/`.

In [ ]:
with EVAL_METRICS_PATH.open(encoding="utf-8") as handle:
    checkpoint_plot_metrics = json.load(handle)

df_checkpoint_plots = pd.DataFrame([
    {
        "checkpoint": row["ckpt_name"],
        "step": row["step"],
        "FID negative": row["negative"]["FID"],
        "FID positive": row["positive"]["FID"],
        "FID medio": row["avg_FID"],
        "precision media": row["avg_precision"],
        "recall media": row["avg_recall"],
    }
    for row in checkpoint_plot_metrics
]).sort_values("step")

best_plot_row = df_checkpoint_plots.loc[df_checkpoint_plots["FID medio"].idxmin()]

fig, ax = plt.subplots(figsize=(10, 5))
for column, marker in [
    ("FID negative", "o"),
    ("FID positive", "s"),
    ("FID medio", "^"),
]:
    ax.plot(df_checkpoint_plots["step"], df_checkpoint_plots[column], marker=marker, label=column)
ax.scatter(
    [best_plot_row["step"]],
    [best_plot_row["FID medio"]],
    s=180,
    facecolors="none",
    edgecolors="red",
    linewidths=2,
    label=f"best: {best_plot_row['checkpoint']}",
)
ax.set(title="FID sul validation set per checkpoint", xlabel="Training step", ylabel="FID")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(PLOTS_DIR / "fid_per_checkpoint.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 6))
scatter = ax.scatter(
    df_checkpoint_plots["recall media"],
    df_checkpoint_plots["precision media"],
    c=df_checkpoint_plots["step"],
    cmap="viridis",
    s=65,
)
ax.scatter(
    [best_plot_row["recall media"]],
    [best_plot_row["precision media"]],
    s=220,
    facecolors="none",
    edgecolors="red",
    linewidths=2,
    label=f"best FID: {best_plot_row['checkpoint']}",
)
ax.set(
    title="PRDC medio per checkpoint",
    xlabel="Recall medio",
    ylabel="Precision media",
)
ax.grid(alpha=0.25)
ax.legend()
fig.colorbar(scatter, ax=ax, label="Training step")
fig.tight_layout()
fig.savefig(PLOTS_DIR / "precision_recall_per_checkpoint.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

print("Grafici checkpoint salvati in:", PLOTS_DIR)

### 5.2 Generazione RAW finale a 100 inference step

La cella successiva usa il checkpoint selezionato sul validation set per costruire il
dataset sintetico RAW finale.

Per ciascuna classe (`positive` e `negative`) vengono prodotte esattamente 2722 immagini (1361 sarebbe il numero necessario per bilanciare positivi e negativi nel train set, ne generiamo il doppio) in `experiments/.../generated_images/final/<class_name>/`. Le 100 immagini sintetiche già generate durante la valutazione del checkpoint selezionato vengono copiate nel dataset RAW finale; la pipeline genera soltanto quelle mancanti. Questo evita una generazione ridondante per ragioni computazionali e di tempo. Si tratta esclusivamente di immagini sintetiche già prodotte dal modello: non vengono introdotte sovrapposizioni tra train, validation e test reali.

Prima e dopo la generazione vengono effettuati controlli sul numero di file e sui duplicati byte-per-byte. Se entrambe le classi sono già complete, la pipeline non viene caricata e l'operazione viene saltata. Un eventuale cambio del checkpoint migliore in presenza di immagini già generate viene segnalato per evitare di mescolare output di modelli diversi.

Il riepilogo della generazione viene aggiornato in `results/2_diffusers/02_sd21_filtered_100steps/metrics/generation_info.json`, mentre le misure ambientali vengono aggiunte a `results/2_diffusers/02_sd21_filtered_100steps/ecotracker/sustainability_generation.jsonl`.

In [ ]:
# Generazione finale con il miglior checkpoint
print(f"Best checkpoint  : {BEST_CHECKPOINT.name}")
print(f"Immagini/classe  : {N_FINAL_IMAGES_PER_CLASS}")
print(f"Output           : {FINAL_GEN_DIR}")

BEST_EVAL_DIR = EVAL_DIR / BEST_CHECKPOINT.name
_class_config = {
    "negative": {
        "prompt": NEGATIVE_PROMPT,
        "final_dir": FINAL_NEG_DIR,
        "eval_dir": BEST_EVAL_DIR / "negative",
        "seed": EVAL_SEED,
        "prefix": "eval_neg",
    },
    "positive": {
        "prompt": POSITIVE_PROMPT,
        "final_dir": FINAL_POS_DIR,
        "eval_dir": BEST_EVAL_DIR / "positive",
        "seed": EVAL_SEED + 1,
        "prefix": "eval_pos",
    },
}

from parallel_generation_utils import SD_SEED_STRATEGY, copy_validated_sd_evaluation_images, final_sd_generation_plan

def _final_plan(cfg):
    return final_sd_generation_plan(Path(cfg["final_dir"]), N_FINAL_IMAGES_PER_CLASS, cfg["prefix"])

_final_already_complete = all(
    _final_plan(_class_config[_cls])["complete"]
    for _cls in FINAL_GENERATE_CLASSES
)
if _final_already_complete:
    print("PNG finali completi: valido comunque lineage e manifest prima dello skip.")
generation_info_path = GENERATION_INFO_PATH
if generation_info_path.exists():
    with open(generation_info_path, encoding="utf-8") as f:
        previous_generation = json.load(f)
    previous_best = previous_generation.get("best_checkpoint")
    final_images_exist = any(any(Path(cfg["final_dir"]).glob("*.png")) for cfg in _class_config.values())
    if previous_best and previous_best != BEST_CHECKPOINT.name and final_images_exist:
        raise RuntimeError(
            f"Il best checkpoint e' cambiato da {previous_best} a {BEST_CHECKPOINT.name}. "
            "Usa una nuova FINAL_GEN_DIR o svuota consapevolmente le cartelle finali."
        )

pipe_final = None  # La generazione finale usa sempre seed per-immagine nel worker isolato.
removed_overlaps = {}

with measure_sustainability(label=f"final_gen_{BEST_CHECKPOINT.name}", sample_interval=0.5) as eco_final:
    for _cls in FINAL_GENERATE_CLASSES:
        cfg = _class_config[_cls]
        eval_available = N_EVAL_IMAGES_PER_CLASS

        print(f"\nClasse {_cls.upper()}")
        print(f"  Target finale              : {N_FINAL_IMAGES_PER_CLASS}")
        print(f"  Immagini eval riutilizzate : {eval_available}")

        # Copia prima le immagini riusate, poi completa il totale.
        reuse_provenance = copy_validated_sd_evaluation_images(
            source_dir=cfg["eval_dir"], final_dir=cfg["final_dir"], count=eval_available,
            reused_prefix=cfg["prefix"], checkpoint_path=str(BEST_CHECKPOINT),
            class_name=_cls, prompt=cfg["prompt"],
            base_seed=EVAL_SEED, num_inference_steps=INFERENCE_STEPS,
            guidance_scale=EVAL_GUIDANCE_SCALE, resolution=RESOLUTION,
            checkpoint_type=SD_CHECKPOINT_TYPE, base_model_dir=LOCAL_MODEL_DIR,
        )
        n_reused = len(reuse_provenance["files"])
        n_new_target = N_FINAL_IMAGES_PER_CLASS - n_reused
        print(f"  Copiate realmente da evaluation: {n_reused}")
        print(f"  Quota gen_* nel dataset finale : {n_new_target}")

        removed = remove_generated_duplicates_of_reused_images(
            final_dir=cfg["final_dir"],
            reused_prefix=cfg["prefix"],
        )
        removed_overlaps[_cls] = len(removed)

        _plan_before = _final_plan(cfg)
        n_before = _plan_before["n_valid_reused"] + len(_plan_before["valid_gen_indices"])
        print(f"  Immagini uniche presenti   : {n_before}")
        print(f"  Nuove da generare          : {max(0, N_FINAL_IMAGES_PER_CLASS - n_before)}")

        _sd_parallel_generate_final(
            BEST_CHECKPOINT, LOCAL_MODEL_DIR, cfg["prompt"], cfg["final_dir"], N_FINAL_IMAGES_PER_CLASS,
            INFERENCE_STEPS, EVAL_GUIDANCE_SCALE, cfg["seed"], RESOLUTION, _cls, cfg["prefix"], n_reused,
        )

        duplicates = duplicate_png_groups(cfg["final_dir"])
        if duplicates:
            raise RuntimeError(f"Trovati {len(duplicates)} gruppi duplicati in {cfg['final_dir']}.")
        _verified_plan = _final_plan(cfg)
        if not _verified_plan["complete"]:
            raise RuntimeError(f"Dataset finale non valido per {_cls}: {_verified_plan}")

del pipe_final
gc.collect()
torch.cuda.empty_cache()

_n_neg = N_FINAL_IMAGES_PER_CLASS if _final_plan(_class_config["negative"])["complete"] else 0
_n_pos = N_FINAL_IMAGES_PER_CLASS if _final_plan(_class_config["positive"])["complete"] else 0
_final_record = eco_final.metrics.to_dict()
_final_record.update({
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "record_type": "final_generation",
    "generation_log_schema": 2,
    "best_checkpoint": BEST_CHECKPOINT.name,
    "selection_reference": "validation",
    "selection_metrics_path": str(EVAL_METRICS_PATH),
    "avg_FID": float(best_row["avg_FID"]),
    "avg_IS_mean": float(best_row["avg_IS_mean"]),
    "n_per_class": N_FINAL_IMAGES_PER_CLASS,
    "n_eval_reused": N_EVAL_IMAGES_PER_CLASS,
    "removed_overlaps": removed_overlaps,
    "n_negative_final": _n_neg,
    "n_positive_final": _n_pos,
    "inference_steps": INFERENCE_STEPS,
    "guidance_scale": EVAL_GUIDANCE_SCALE,
    "generated_classes": FINAL_GENERATE_CLASSES,
    "generated_negative": str(FINAL_NEG_DIR),
    "generated_positive": str(FINAL_POS_DIR),
    "status": "completed",
    "seed_strategy": SD_SEED_STRATEGY,
    "parallel_generation": PARALLEL_GENERATION,
    "generation_gpu_devices": GENERATION_GPU_DEVICES,
})

with open(generation_info_path, "w", encoding="utf-8") as f:
    json.dump(_final_record, f, indent=2, ensure_ascii=False)
with open(FINAL_GENERATION_LOG, "a", encoding="utf-8") as f:
    f.write(json.dumps(_final_record, ensure_ascii=False) + "\n")

print(f"\nMetriche eco-tracking generazione finale:\n{eco_final.metrics}")
print(
    f"Immagini finali — negative: {_n_neg} -> {FINAL_NEG_DIR}\n"
    f"Immagini finali — positive: {_n_pos} -> {FINAL_POS_DIR}"
)

### 5.3 Dataset RAW matched a 100 inference step

Per misurare l'effetto del filtro in modo diretto viene creato un sottocampione RAW deterministico a 100 inference step con la stessa numerosità delle immagini filtrate: 1361 immagini per classe.

Il sottocampione viene estratto dalle 2722 RAW complete della classe corrispondente, senza modificare le directory RAW originali. La selezione ordina inizialmente i PNG, usa un seed deterministico diverso per classe (`RAW_MATCHED_SEED + label`), copia i file in `generated_images/raw_matched_1361/<class_name>/` e scrive un manifest JSON per ricostruire e validare la selezione. Lo skip è consentito soltanto quando manifest, directory sorgente, numerosità, seed e metodo di selezione sono compatibili.

In [ ]:
# Creazione del dataset RAW matched a 100 inference step.
RAW_MATCHED_SELECTION_METHOD = "sorted_png_without_replacement_numpy_default_rng_v1"
RAW_COMPLETE_COUNT_PER_CLASS = N_FINAL_IMAGES_PER_CLASS


def validate_png_file(path):
    path = Path(path)
    try:
        with Image.open(path) as image:
            image.verify()
    except Exception as exc:
        raise RuntimeError(f"PNG non valido: {path}") from exc
    return True


def png_names_digest(paths):
    digest = hashlib.sha256()
    for path in paths:
        digest.update(path.name.encode("utf-8"))
        digest.update(b"\0")
    return digest.hexdigest()


def deterministic_raw_matched_selection(raw_dir, label):
    raw_dir = Path(raw_dir)
    from parallel_generation_utils import GENERATED_PNG_PATTERN, readable_png_paths
    raw_paths = readable_png_paths(raw_dir, GENERATED_PNG_PATTERN)
    if len(raw_paths) != RAW_COMPLETE_COUNT_PER_CLASS:
        raise RuntimeError(
            f"RAW complete non coerenti in {raw_dir}: "
            f"{len(raw_paths)} != {RAW_COMPLETE_COUNT_PER_CLASS}."
        )
    for path in raw_paths:
        validate_png_file(path)

    random_seed = RAW_MATCHED_SEED + int(label)
    rng = np.random.default_rng(random_seed)
    selected_indices = sorted(
        int(index)
        for index in rng.choice(len(raw_paths), size=RAW_MATCHED_COUNT, replace=False)
    )
    selected_paths = [raw_paths[index] for index in selected_indices]
    selected_names = [path.name for path in selected_paths]
    if len(selected_names) != len(set(selected_names)):
        raise RuntimeError("La selezione RAW matched contiene nomi duplicati.")
    return raw_paths, selected_paths, selected_names, random_seed


def load_json_if_present(path):
    path = Path(path)
    if not path.is_file():
        return None
    try:
        with path.open(encoding="utf-8") as handle:
            return json.load(handle)
    except (OSError, json.JSONDecodeError):
        return None


def raw_matched_manifest_compatible(payload, class_name, label, raw_dir, output_dir, raw_paths, selected_names, random_seed):
    if not payload:
        return False
    expected = {
        "schema_version": 1,
        "experiment_name": EXPERIMENT_NAME,
        "class_name": class_name,
        "label": int(label),
        "source_raw_dir": str(Path(raw_dir)),
        "output_dir": str(Path(output_dir)),
        "n_raw_available": RAW_COMPLETE_COUNT_PER_CLASS,
        "n_selected": RAW_MATCHED_COUNT,
        "random_seed": int(random_seed),
        "selection_method": RAW_MATCHED_SELECTION_METHOD,
        "source_raw_names_sha256": png_names_digest(raw_paths),
    }
    for key, expected_value in expected.items():
        if payload.get(key) != expected_value:
            return False
    return payload.get("selected_names") == selected_names


def ensure_raw_matched_dataset(class_name):
    label = FINAL_CLASS_LABELS[class_name]
    raw_dir = FINAL_DIRS[class_name]
    output_dir = RAW_MATCHED_DIRS[class_name]
    manifest_path = RAW_MATCHED_MANIFEST_PATHS[class_name]
    raw_paths, selected_paths, selected_names, random_seed = deterministic_raw_matched_selection(
        raw_dir=raw_dir,
        label=label,
    )

    payload = load_json_if_present(manifest_path)
    output_paths = [output_dir / name for name in selected_names]
    can_skip = (
        raw_matched_manifest_compatible(
            payload, class_name, label, raw_dir, output_dir, raw_paths, selected_names, random_seed
        )
        and count_pngs(output_dir) == RAW_MATCHED_COUNT
        and all(path.is_file() for path in output_paths)
        and not duplicate_png_groups(output_dir)
    )
    if can_skip:
        for path in output_paths:
            validate_png_file(path)
        print(f"RAW matched {class_name}: manifest compatibile, skip")
        return payload

    output_dir.mkdir(parents=True, exist_ok=True)
    for path in output_dir.glob("*.png"):
        path.unlink()
    for source_path in selected_paths:
        shutil.copy2(source_path, output_dir / source_path.name)

    if count_pngs(output_dir) != RAW_MATCHED_COUNT:
        raise RuntimeError(f"RAW matched {class_name} incompleto: {output_dir}")
    if duplicate_png_groups(output_dir):
        raise RuntimeError(f"RAW matched {class_name} contiene duplicati byte-per-byte.")
    for path in output_paths:
        validate_png_file(path)

    manifest = {
        "schema_version": 1,
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "experiment_name": EXPERIMENT_NAME,
        "class_name": class_name,
        "label": int(label),
        "source_raw_dir": str(raw_dir),
        "output_dir": str(output_dir),
        "n_raw_available": len(raw_paths),
        "n_selected": len(selected_paths),
        "random_seed": int(random_seed),
        "selection_method": RAW_MATCHED_SELECTION_METHOD,
        "source_raw_names_sha256": png_names_digest(raw_paths),
        "selected_names": selected_names,
    }
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    with manifest_path.open("w", encoding="utf-8") as handle:
        json.dump(manifest, handle, indent=2, ensure_ascii=False)
    print(f"RAW matched {class_name}: create {len(selected_paths)} immagini -> {output_dir}")
    print("Manifest:", manifest_path)
    return manifest


raw_matched_manifests = {
    class_name: ensure_raw_matched_dataset(class_name)
    for class_name in FINAL_GENERATE_CLASSES
}

### 5.4 Filtro adattivo

Le immagini sintetiche RAW vengono sottoposte al filtro definito nel modulo condiviso `adaptive_mammography_filter.py`.

Il modulo calcola, sulle immagini reali del training set appartenenti alla classe considerata, statistiche robuste relative a:

* proporzione di foreground;
* intensità media e deviazione standard del tessuto;
* entropia;
* varianza del Laplaciano;
* compattezza della componente principale.

Ogni immagine sintetica viene quindi analizzata mediante le stesse feature. Le immagini che non rispettano i criteri morfologici e di qualità vengono rifiutate; quelle accettate ricevono uno score basato sulla distanza robusta dalle statistiche delle immagini reali.

Le candidate accettate vengono ordinate per score decrescente e vengono selezionate le prime **1361 immagini per classe**.

Nel presente notebook `run_adaptive_filter.py` richiama le funzioni condivise del filtro e gestisce gli aspetti specifici del ramo Stable Diffusion:

* nomi storici delle immagini filtrate;
* directory di output;
* report CSV;
* summary JSON;
* controllo degli artefatti già presenti.

Il filtro viene calibrato separatamente per la classe positiva e per la classe negativa, utilizzando esclusivamente le immagini reali del training set della classe corrispondente.

Configurazione del presente esperimento:

* **classe positiva:** 2722 immagini RAW → 1361 immagini filtrate;
* **classe negativa:** 2722 immagini RAW → 1361 immagini filtrate;
* **soglia iniziale del foreground:** `nonblack_threshold = 10`.

In [ ]:
def file_signature(path):
    path = Path(path)
    return {
        "path": str(path),
        "size": path.stat().st_size,
        "mtime_ns": path.stat().st_mtime_ns,
    }


def png_files_signature(directory):
    return [
        {
            "name": path.name,
            "size": path.stat().st_size,
            "mtime_ns": path.stat().st_mtime_ns,
        }
        for path in sorted(Path(directory).glob("*.png"))
    ]


def evaluate_generated_dir_against_split(generated_dir, metadata_path, label, n_images, seed):
    generated_dir = Path(generated_dir)
    n_generated = count_pngs(generated_dir)
    if n_generated == 0:
        raise RuntimeError(f"Nessuna immagine generata in {generated_dir}.")
    duplicates = duplicate_png_groups(generated_dir)
    if duplicates:
        raise RuntimeError(f"Trovati {len(duplicates)} gruppi duplicati in {generated_dir}.")

    with temporary_real_reference_dir_from_split_metadata(
        metadata_path=metadata_path,
        label=label,
        n_images=n_images,
        seed=seed,
    ) as real_reference_dir:
        evaluator = GenerativeEvaluator(
            real_dir=real_reference_dir,
            generated_dir=generated_dir,
            batch_size=8,
            num_workers=0,
        )
        fid_is_metrics, real_features, fake_features = evaluator.compute_with_features()
        prdc_metrics = compute_prdc_metrics(real_features, fake_features)

    return {
        **fid_is_metrics,
        **prdc_metrics,
        "n_generated": n_generated,
    }

In [ ]:
# IDEMPOTENT_GUARD_V1:filter
if phase_should_run(PHASE_PLAN, "filter", PLAN_ONLY):
    # Filtra entrambe le classi con calibrazione indipendente.
    filter_summaries = {}
    filter_script = UTILITY_DIR / "run_adaptive_filter.py"
    if not filter_script.is_file():
        raise FileNotFoundError(f"Script filtro non trovato: {filter_script}")

    for class_name in FINAL_GENERATE_CLASSES:
        class_label = FINAL_CLASS_LABELS[class_name]
        raw_dir = FINAL_DIRS[class_name]
        filtered_dir = FILTERED_DIRS[class_name]
        report_path = FILTER_REPORT_PATHS[class_name]
        summary_path = FILTER_SUMMARY_PATHS[class_name]

        command = [
            sys.executable,
            str(filter_script),
            "--class-name", class_name,
            "--label", str(class_label),
            "--raw-dir", str(raw_dir),
            "--filtered-dir", str(filtered_dir),
            "--n-selected", str(N_SELECTED_PER_CLASS),
            "--train-metadata", str(TRAIN_METADATA_PATH),
            "--data-processed-dir", str(DATA_PROCESSED_DIR),
            "--report-csv", str(report_path),
            "--summary-json", str(summary_path),
            "--experiment-name", EXPERIMENT_NAME,
            "--eval-seed", str(EVAL_SEED),
            "--nonblack-threshold", str(NONBLACK_THRESHOLD),
        ]
        subprocess.run(command, check=True)

        if not summary_path.is_file():
            raise FileNotFoundError(f"Summary filtro non trovato per {class_name}: {summary_path}")
        with summary_path.open("r", encoding="utf-8") as handle:
            filter_summaries[class_name] = json.load(handle)

        if count_pngs(filtered_dir) != N_SELECTED_PER_CLASS:
            raise RuntimeError(f"Numero immagini filtrate non valido per {class_name}.")
        if duplicate_png_groups(filtered_dir):
            raise RuntimeError(f"Il dataset filtrato {class_name} contiene duplicati.")

    print("\nSummary filtro adattivo:")
    for class_name, summary in filter_summaries.items():
        print(f"{class_name}: {summary.get('status', 'unknown')} -> {FILTERED_DIRS[class_name]}")

### 5.5 Confronto RAW vs filtrate sul validation set

La valutazione sul validation set viene utilizzata per analizzare l’effetto del filtro senza coinvolgere il test set.

Per ciascuna classe vengono considerati tre dataset prodotti a 100 inference step:

* **RAW complete (`n = 2722`)**: insieme completo delle immagini generate, mantenuto come riferimento descrittivo;
* **RAW matched (`n = 1361`)**: sottocampione casuale deterministico delle RAW complete, con la stessa numerosità del dataset filtrato;
* **filtrate (`n = 1361`)**: immagini selezionate mediante il filtro adattivo.

Il confronto metodologicamente diretto dell’effetto del filtro è:

**100 step RAW matched (`n = 1361`) vs 100 step filtrate (`n = 1361`)**

In questo confronto il numero di immagini rimane invariato e cambia il criterio con cui viene costruito il dataset:

* il dataset RAW matched è ottenuto mediante campionamento casuale senza reinserimento;
* il dataset filtrato contiene le immagini che superano i controlli morfologici e qualitativi e che presentano gli score migliori rispetto alle statistiche delle immagini reali del training set.

Le 2722 RAW complete rimangono disponibili nei risultati con ruolo `context_only` e non vengono utilizzate come unica baseline per quantificare l’effetto del filtro.

Le metriche considerate sono:

* **FID**;
* **Inception Score**;
* **precision**;
* **recall**;
* **density**;
* **coverage**.

Il filtro può aumentare la fedeltà rispetto alla distribuzione reale, migliorando metriche come FID, precision o density, ma può contemporaneamente ridurre parte della varietà rappresentata, con possibili diminuzioni di recall o coverage.

I risultati vengono quindi interpretati considerando il compromesso tra:

* **fedeltà dei campioni sintetici**;
* **diversità della distribuzione generata**.

In [ ]:
metric_columns = [
    "FID", "IS_mean", "IS_std", "precision", "recall", "density", "coverage"
]
validation_sets = {}
for class_name in FINAL_GENERATE_CLASSES:
    label = FINAL_CLASS_LABELS[class_name]
    validation_sets[f"{class_name}_raw_complete_{N_FINAL_IMAGES_PER_CLASS}"] = {
        "class": class_name,
        "label": label,
        "stage": "raw",
        "comparison_role": "context_only",
        "generated_dir": FINAL_DIRS[class_name],
        "expected_size": N_FINAL_IMAGES_PER_CLASS,
    }
    validation_sets[f"{class_name}_raw_matched_{RAW_MATCHED_COUNT}"] = {
        "class": class_name,
        "label": label,
        "stage": "raw_matched",
        "comparison_role": "filter_comparison",
        "generated_dir": RAW_MATCHED_DIRS[class_name],
        "expected_size": RAW_MATCHED_COUNT,
    }
    validation_sets[f"{class_name}_filtered_{N_SELECTED_PER_CLASS}_adaptive_mask"] = {
        "class": class_name,
        "label": label,
        "stage": "filtered",
        "comparison_role": "filter_comparison",
        "generated_dir": FILTERED_DIRS[class_name],
        "expected_size": N_SELECTED_PER_CLASS,
    }
validation_eval_config = {
    "schema_version": 2,
    "metric_backend": "generative_evaluator.py",
    "reference_split": "validation",
    "n_validation_per_class": N_VALIDATION_IMAGES_PER_CLASS,
    "knn_k": PRDC_NEAREST_K,
    "inception_batch": 8,
    "classes": FINAL_GENERATE_CLASSES,
    "sets": {
        set_name: {
            "class": spec["class"],
            "label": int(spec["label"]),
            "stage": spec["stage"],
            "comparison_role": spec["comparison_role"],
            "expected_size": int(spec["expected_size"]),
        }
        for set_name, spec in validation_sets.items()
    },
}
validation_input_signature = {
    "sets": {
        set_name: png_files_signature(spec["generated_dir"])
        for set_name, spec in validation_sets.items()
    },
    "raw_matched_manifests": {
        class_name: file_signature(path) if Path(path).is_file() else None
        for class_name, path in RAW_MATCHED_MANIFEST_PATHS.items()
    },
    "validation_csv": file_signature(VALIDATION_METADATA_PATH),
}

use_validation_cache = False
if VALIDATION_COMPARISON_JSON.exists() and not FORCE_RECOMPUTE_VALIDATION_COMPARISON:
    try:
        with VALIDATION_COMPARISON_JSON.open("r", encoding="utf-8") as handle:
            validation_payload = json.load(handle)
        use_validation_cache = (
            validation_payload.get("schema_version") == 2
            and validation_payload.get("config") == validation_eval_config
            and validation_payload.get("input_signature") == validation_input_signature
            and VALIDATION_COMPARISON_CSV.exists()
        )
    except Exception as exc:
        print(f"Cache validation non leggibile, ricalcolo: {exc}")

if use_validation_cache:
    df_validation_comparison = pd.read_csv(VALIDATION_COMPARISON_CSV)
    print("Metriche validation caricate da cache:", VALIDATION_COMPARISON_JSON)
else:
    comparison_rows = []
    for set_name, spec in validation_sets.items():
        class_name = spec["class"]
        label = spec["label"]
        generated_dir = spec["generated_dir"]
        expected_size = spec["expected_size"]
        n_generated = count_pngs(generated_dir)
        if n_generated != expected_size:
            raise RuntimeError(
                f"Dataset {set_name} incompleto: {n_generated} != {expected_size}."
            )
        print(f"Valutazione validation: {set_name}")
        metrics = evaluate_generated_dir_against_split(
            generated_dir=generated_dir,
            metadata_path=VALIDATION_METADATA_PATH,
            label=label,
            n_images=N_VALIDATION_IMAGES_PER_CLASS,
            seed=EVAL_SEED + label,
        )
        comparison_rows.append({
            "class": class_name,
            "label": int(label),
            "set_name": set_name,
            "stage": spec["stage"],
            "comparison_role": spec["comparison_role"],
            **metrics,
        })

    df_validation_comparison = pd.DataFrame(comparison_rows)[
        [
            "class", "label", "set_name", "stage", "comparison_role",
            *metric_columns, "n_generated",
        ]
    ]
    df_validation_comparison.to_csv(VALIDATION_COMPARISON_CSV, index=False)
    with VALIDATION_COMPARISON_JSON.open("w", encoding="utf-8") as handle:
        json.dump({
            "schema_version": 2,
            "config": validation_eval_config,
            "input_signature": validation_input_signature,
            "timestamp": datetime.now().isoformat(timespec="seconds"),
            "experiment": EXPERIMENT_NAME,
            "evaluation_reference": "validation",
            "metrics": df_validation_comparison.to_dict(orient="records"),
        }, handle, indent=2, ensure_ascii=False)

print("\nConfronto a 100 step su validation:")
print(df_validation_comparison.to_string(index=False))
print("\nSalvato in:", VALIDATION_COMPARISON_CSV)

#### 5.5a Griglie qualitative dei campioni filtrati

Le griglie seguenti mostrano campioni filtrati positivi e negativi. Il confronto numerico principale viene sintetizzato nella sezione successiva con `filter_before_after.png` e `filter_oriented_percent_delta.png`.

In [ ]:
for class_name in FINAL_GENERATE_CLASSES:
    sample_dir = FILTERED_DIRS[class_name]
    image_paths = sorted(sample_dir.glob("*.png"))
    if not image_paths:
        print(f"Nessuna immagine disponibile per la griglia {class_name}.")
        continue

    n_samples = min(16, len(image_paths))
    sample_indices = np.linspace(0, len(image_paths) - 1, n_samples, dtype=int)
    selected_paths = [image_paths[index] for index in sample_indices]

    fig, axes = plt.subplots(4, 4, figsize=(10, 10))
    for ax, image_path in zip(axes.flat, selected_paths):
        with Image.open(image_path) as image:
            ax.imshow(image.convert("L"), cmap="gray")
        ax.set_title(image_path.name, fontsize=7)
        ax.axis("off")
    for ax in axes.flat[n_samples:]:
        ax.axis("off")
    fig.suptitle(f"Campioni generati filtrati - {class_name}")
    fig.tight_layout()
    output_path = PLOTS_DIR / f"samples_{class_name}.png"
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)

print("Griglie filtrate salvate in:", PLOTS_DIR)

#### 5.5b Trade-off introdotto dal filtro adattivo

Questa sezione usa esclusivamente artefatti già salvati, senza ricalcolare metriche. Il confronto prima/dopo legge `validation_comparison_100_steps_raw_matched_vs_filtered.csv` e usa come baseline diretta le **100 step RAW matched (n=1361)**, non le RAW complete da 2722 immagini.

Sono prodotti due plot principali:

- `filter_before_after.png`, per i valori assoluti delle metriche;
- `filter_oriented_percent_delta.png`, per le variazioni percentuali orientate al miglioramento.

In [ ]:
# Sorgenti in sola lettura dell'esperimento corrente a 100 inference step.
FILTER_COMPARISON_CSV = VALIDATION_COMPARISON_CSV
FILTER_FINAL_TEST_JSON = FINAL_TEST_METRICS_PATH


def show_and_save_filter_figure(figure, filename):
    output_path = PLOTS_DIR / filename
    figure.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.show()
    plt.close(figure)
    print("Salvato:", output_path)


if not FILTER_COMPARISON_CSV.is_file():
    print(
        "Confronto RAW matched/filtrate non disponibile: grafici non generati. File atteso:",
        FILTER_COMPARISON_CSV,
    )
else:
    filter_comparison = pd.read_csv(FILTER_COMPARISON_CSV)
    required_columns = {
        "class", "stage", "FID", "IS_mean", "precision", "recall", "density", "coverage"
    }
    missing_columns = required_columns - set(filter_comparison.columns)

    if missing_columns:
        print(
            "Il CSV non contiene ancora tutte le metriche richieste:",
            sorted(missing_columns),
        )
    else:
        filter_comparison = filter_comparison.loc[
            filter_comparison["stage"].isin(["raw_matched", "filtered"])
        ].copy()
        filter_comparison["variant"] = filter_comparison["stage"].map({
            "raw_matched": "100 step RAW matched (n=1361)",
            "filtered": "100 step filtrate (n=1361)",
        })
        filter_classes = ["positive", "negative"]
        class_colors = {"positive": "#1f77b4", "negative": "#ff7f0e"}

        final_test_context = None
        if FILTER_FINAL_TEST_JSON.is_file():
            with FILTER_FINAL_TEST_JSON.open(encoding="utf-8") as handle:
                final_test_context = json.load(handle)
        else:
            print(
                "Metriche finali sul test non disponibili; "
                "i grafici useranno soltanto il validation:",
                FILTER_FINAL_TEST_JSON,
            )

        def metric_pair(class_name, metric):
            class_rows = filter_comparison.loc[filter_comparison["class"] == class_name]
            values = class_rows.set_index("stage")[metric]
            if not {"raw_matched", "filtered"} <= set(values.index):
                raise ValueError(
                    f"Confronto RAW matched/filtrate incompleto per {class_name}, metrica {metric}."
                )
            return float(values["raw_matched"]), float(values["filtered"])

        def percentage_delta(reference_value, compared_value):
            """Variazione percentuale; non definita quando la baseline e' zero."""
            if np.isclose(reference_value, 0.0):
                return np.nan
            return (compared_value - reference_value) / reference_value * 100

        def percentage_delta_label(delta):
            return "n.d. (RAW=0)" if not np.isfinite(delta) else f"{delta:+.1f}%"

        metric_specs = [
            ("FID", "FID - più basso è meglio"),
            ("IS_mean", "Inception Score - più alto indica maggiore varietà"),
            ("precision", "Precision PRDC - più alto è meglio"),
            ("recall", "Recall PRDC - più alto indica maggiore copertura"),
        ]
        figure, axes = plt.subplots(2, 2, figsize=(13, 10), constrained_layout=True)
        for axis, (metric, title) in zip(axes.flat, metric_specs):
            for class_name in filter_classes:
                raw_value, filtered_value = metric_pair(class_name, metric)
                actual_delta = percentage_delta(raw_value, filtered_value)
                axis.plot(
                    [0, 1],
                    [raw_value, filtered_value],
                    color=class_colors[class_name],
                    marker="o",
                    linewidth=2.2,
                    markersize=7,
                    label=class_name,
                )
                axis.annotate(
                    "",
                    xy=(1, filtered_value),
                    xytext=(0, raw_value),
                    arrowprops={
                        "arrowstyle": "-|>",
                        "color": class_colors[class_name],
                        "linewidth": 2.2,
                        "mutation_scale": 13,
                    },
                )
                axis.annotate(
                    percentage_delta_label(actual_delta),
                    xy=(1, filtered_value),
                    xytext=(7, 0),
                    textcoords="offset points",
                    va="center",
                    fontsize=9,
                    color=class_colors[class_name],
                )

            axis.set_xticks([0, 1], ["RAW matched", "Filtrate"])
            axis.set_title(title)
            axis.grid(axis="y", alpha=0.25)

        axes[0, 0].legend(title="Classe")
        figure.suptitle(
            "Effetto del filtro adattivo: 100 step RAW matched vs 100 step filtrate",
            fontsize=15,
        )
        if final_test_context is not None:
            test_context_text = (
                "Contesto test sulle sole filtrate: "
                f"FID medio={final_test_context.get('avg_FID', float('nan')):.2f}, "
                f"precision media={final_test_context.get('avg_precision', float('nan')):.3f}, "
                f"recall media={final_test_context.get('avg_recall', float('nan')):.3f}"
            )
            figure.text(0.5, -0.015, test_context_text, ha="center", fontsize=9)
        show_and_save_filter_figure(figure, "filter_before_after.png")

        delta_metrics = ["FID", "IS_mean", "precision", "recall", "density", "coverage"]
        oriented_delta_rows = []
        for class_name in filter_classes:
            for metric in delta_metrics:
                raw_value, filtered_value = metric_pair(class_name, metric)
                raw_delta = percentage_delta(raw_value, filtered_value)
                oriented_delta = -raw_delta if metric == "FID" else raw_delta
                oriented_delta_rows.append({
                    "class": class_name,
                    "metric": metric,
                    "oriented_delta": oriented_delta,
                })

        delta_frame = pd.DataFrame(oriented_delta_rows)
        figure, axis = plt.subplots(figsize=(11, 7), constrained_layout=True)
        y_positions = np.arange(len(delta_metrics))
        bar_height = 0.34
        class_offsets = {"positive": -bar_height / 2, "negative": bar_height / 2}
        class_hatches = {"positive": "//", "negative": "\\"}

        for class_name in filter_classes:
            class_delta = (
                delta_frame.loc[delta_frame["class"] == class_name]
                .set_index("metric")
                .reindex(delta_metrics)["oriented_delta"]
            )
            positions = y_positions + class_offsets[class_name]
            colors = [
                "#808080" if not np.isfinite(value) else ("#2e8b57" if value >= 0 else "#c0392b")
                for value in class_delta
            ]
            bars = axis.barh(
                positions,
                class_delta,
                height=bar_height,
                color=colors,
                edgecolor="white",
                hatch=class_hatches[class_name],
            )
            axis.bar_label(bars, labels=[percentage_delta_label(value) for value in class_delta], padding=3)

        axis.axvline(0, color="black", linewidth=1)
        axis.set_yticks(y_positions, ["FID (segno invertito)", "IS_mean", "Precision", "Recall", "Density", "Coverage"])
        axis.set_xlabel("Variazione percentuale orientata al miglioramento")
        axis.set_title("Delta del filtro: 100 step RAW matched vs filtrate")
        axis.grid(axis="x", alpha=0.25)
        axis.legend(handles=[
            Patch(facecolor="#2e8b57", label="Miglioramento"),
            Patch(facecolor="#c0392b", label="Riduzione / trade-off"),
            Patch(facecolor="white", edgecolor="black", hatch="//", label="Positive"),
            Patch(facecolor="white", edgecolor="black", hatch="\\", label="Negative"),
        ], loc="best")
        axis.text(
            0.5,
            -0.13,
            "Per il FID il segno è invertito: una barra positiva indica che il valore è diminuito. "
            "Le riduzioni di IS e recall descrivono il trade-off su varietà e copertura.",
            transform=axis.transAxes,
            ha="center",
            va="top",
            fontsize=9,
        )
        show_and_save_filter_figure(figure, "filter_oriented_percent_delta.png")

### 5.6 Test finale a 100 inference step

La valutazione finale sul test set mantiene separati tre ruoli:

- **RAW complete a 100 step**: 2722 immagini per classe, usate solo come contesto descrittivo nel confronto sul validation set salvato in `validation_comparison_100_steps_raw_matched_vs_filtered.*`;
- **RAW matched a 100 step**: 1361 immagini per classe, baseline diretta per misurare l'effetto del filtro, salvata in `final_test_metrics_raw_matched_100_steps.json`;
- **filtrate a 100 step**: 1361 immagini per classe, output del filtro adattivo, salvate in `final_test_metrics.json`.

Le celle non ricalcolano nulla quando una cache compatibile è già presente. Se le metriche RAW matched non esistono, la cella dedicata è pronta per essere eseguita manualmente e calcolerà FID, Inception Score e PRDC contro il test set.

In [ ]:
# Test finale RAW matched a 100 inference step: baseline diretta del filtro.
raw_matched_metric_columns = [
    "FID", "IS_mean", "IS_std", "precision", "recall", "density", "coverage"
]
raw_matched_eval_config = {
    "schema_version": 1,
    "metric_backend": "generative_evaluator.py",
    "reference_split": "test",
    "stage": "raw_matched",
    "inference_steps": INFERENCE_STEPS,
    "source_n_raw": N_FINAL_IMAGES_PER_CLASS,
    "target_n": RAW_MATCHED_COUNT,
    "n_test_per_class": N_TEST_IMAGES_PER_CLASS,
    "knn_k": PRDC_NEAREST_K,
    "inception_batch": 8,
    "classes": FINAL_GENERATE_CLASSES,
    "raw_matched_dirs": {
        class_name: str(RAW_MATCHED_DIRS[class_name])
        for class_name in FINAL_GENERATE_CLASSES
    },
    "raw_matched_manifests": {
        class_name: str(RAW_MATCHED_MANIFEST_PATHS[class_name])
        for class_name in FINAL_GENERATE_CLASSES
    },
}
raw_matched_input_signature = {
    "classes": {
        class_name: png_files_signature(RAW_MATCHED_DIRS[class_name])
        for class_name in FINAL_GENERATE_CLASSES
    },
    "raw_matched_manifests": {
        class_name: file_signature(path) if Path(path).is_file() else None
        for class_name, path in RAW_MATCHED_MANIFEST_PATHS.items()
    },
    "test_csv": file_signature(TEST_METADATA_PATH),
}

use_raw_matched_cache = False
if FINAL_TEST_RAW_MATCHED_METRICS_PATH.exists() and not FORCE_RECOMPUTE_FINAL_TEST:
    try:
        with FINAL_TEST_RAW_MATCHED_METRICS_PATH.open("r", encoding="utf-8") as handle:
            raw_matched_test_summary = json.load(handle)
        use_raw_matched_cache = (
            raw_matched_test_summary.get("schema_version") == 1
            and raw_matched_test_summary.get("config") == raw_matched_eval_config
            and raw_matched_test_summary.get("input_signature") == raw_matched_input_signature
            and FINAL_TEST_RAW_MATCHED_METRICS_CSV.exists()
        )
    except Exception as exc:
        print(f"Cache test RAW matched non leggibile, ricalcolo: {exc}")

if use_raw_matched_cache:
    df_raw_matched_test = pd.read_csv(FINAL_TEST_RAW_MATCHED_METRICS_CSV)
    raw_matched_test_metrics = raw_matched_test_summary.get("per_class", {})
    print("TEST RAW MATCHED: metriche caricate da cache:", FINAL_TEST_RAW_MATCHED_METRICS_PATH)
else:
    raw_matched_test_metrics = {}
    print("TEST RAW MATCHED: dataset RAW matched contro i reali del test")
    print("Checkpoint scelto sul validation:", BEST_CHECKPOINT.name)

    for class_name in FINAL_GENERATE_CLASSES:
        label = FINAL_CLASS_LABELS[class_name]
        generated_dir = RAW_MATCHED_DIRS[class_name]
        manifest_path = RAW_MATCHED_MANIFEST_PATHS[class_name]
        if not manifest_path.is_file():
            raise FileNotFoundError(
                f"Manifest RAW matched mancante per {class_name}: {manifest_path}. "
                "Esegui prima la cella di creazione del dataset RAW matched."
            )
        with manifest_path.open(encoding="utf-8") as handle:
            manifest = json.load(handle)
        n_generated = count_pngs(generated_dir)
        if n_generated != RAW_MATCHED_COUNT:
            raise RuntimeError(
                f"Dataset RAW matched incompleto per {class_name}: "
                f"{n_generated} != {RAW_MATCHED_COUNT}."
            )
        if duplicate_png_groups(generated_dir):
            raise RuntimeError(f"Il dataset RAW matched {class_name} contiene duplicati.")

        metrics = evaluate_generated_dir_against_split(
            generated_dir=generated_dir,
            metadata_path=TEST_METADATA_PATH,
            label=label,
            n_images=N_TEST_IMAGES_PER_CLASS,
            seed=EVAL_SEED + label,
        )
        raw_matched_test_metrics[class_name] = {
            **metrics,
            "inference_steps": INFERENCE_STEPS,
            "stage": "raw_matched",
            "sampling_seed": RAW_MATCHED_SEED + int(label),
            "source_n_raw": N_FINAL_IMAGES_PER_CLASS,
            "target_n": RAW_MATCHED_COUNT,
            "class_name": class_name,
            "label": int(label),
            "checkpoint_path": str(BEST_CHECKPOINT),
            "checkpoint_id": BEST_CHECKPOINT.name,
            "manifest": str(manifest_path),
            "manifest_payload": manifest,
            "real_label": int(label),
            "n_real_reference": N_TEST_IMAGES_PER_CLASS,
            "real_reference_metadata": str(TEST_METADATA_PATH),
            "generated_dir": str(generated_dir),
        }

    n_classes = len(raw_matched_test_metrics)
    averages = {
        f"avg_{metric}": round(
            sum(row[metric] for row in raw_matched_test_metrics.values()) / n_classes,
            4,
        )
        for metric in raw_matched_metric_columns
    }
    raw_matched_test_summary = {
        "schema_version": 1,
        "config": raw_matched_eval_config,
        "input_signature": raw_matched_input_signature,
        "experiment": EXPERIMENT_NAME,
        "best_checkpoint": BEST_CHECKPOINT.name,
        "checkpoint_path": str(BEST_CHECKPOINT),
        "inference_steps": INFERENCE_STEPS,
        "stage": "raw_matched",
        "n_generated": RAW_MATCHED_COUNT,
        "sampling_seed": RAW_MATCHED_SEED,
        "source_n_raw": N_FINAL_IMAGES_PER_CLASS,
        "target_n": RAW_MATCHED_COUNT,
        "evaluation_reference": "test",
        "classes_evaluated": list(raw_matched_test_metrics),
        **averages,
        "per_class": raw_matched_test_metrics,
        "timestamp": datetime.now().isoformat(timespec="seconds"),
    }
    with FINAL_TEST_RAW_MATCHED_METRICS_PATH.open("w", encoding="utf-8") as handle:
        json.dump(raw_matched_test_summary, handle, indent=2, ensure_ascii=False)

    raw_matched_rows = [
        {"class": class_name, **metrics}
        for class_name, metrics in raw_matched_test_metrics.items()
    ]
    raw_matched_rows.append({
        "class": "average",
        **{metric: averages[f"avg_{metric}"] for metric in raw_matched_metric_columns},
        "n_generated": sum(row["n_generated"] for row in raw_matched_test_metrics.values()),
        "inference_steps": INFERENCE_STEPS,
        "stage": "raw_matched",
        "class_name": "average",
        "label": "",
        "real_label": "",
        "n_real_reference": N_TEST_IMAGES_PER_CLASS,
        "real_reference_metadata": str(TEST_METADATA_PATH),
        "generated_dir": "",
    })
    df_raw_matched_test = pd.DataFrame(raw_matched_rows)
    df_raw_matched_test.to_csv(FINAL_TEST_RAW_MATCHED_METRICS_CSV, index=False)

print("\nMetriche RAW matched sul test:")
print(df_raw_matched_test.to_string(index=False))
print("\nSalvate in:", FINAL_TEST_RAW_MATCHED_METRICS_PATH)

In [ ]:
final_metric_columns = [
    "FID", "IS_mean", "IS_std", "precision", "recall", "density", "coverage"
]
final_eval_config = {
    "metric_backend": "generative_evaluator.py",
    "reference_split": "test",
    "n_test_per_class": N_TEST_IMAGES_PER_CLASS,
    "knn_k": PRDC_NEAREST_K,
    "inception_batch": 8,
    "is_splits": 10,
    "classes": FINAL_GENERATE_CLASSES,
    "filtered_dirs": {
        class_name: str(FILTERED_DIRS[class_name])
        for class_name in FINAL_GENERATE_CLASSES
    },
}
final_input_signature = {
    "classes": {
        class_name: png_files_signature(FILTERED_DIRS[class_name])
        for class_name in FINAL_GENERATE_CLASSES
    },
    "test_csv": file_signature(TEST_METADATA_PATH),
}

use_final_cache = False
if FINAL_TEST_METRICS_PATH.exists() and not FORCE_RECOMPUTE_FINAL_TEST:
    try:
        with FINAL_TEST_METRICS_PATH.open("r", encoding="utf-8") as handle:
            final_test_summary = json.load(handle)
        use_final_cache = (
            final_test_summary.get("schema_version") == 2
            and final_test_summary.get("config") == final_eval_config
            and final_test_summary.get("input_signature") == final_input_signature
            and FINAL_TEST_METRICS_CSV.exists()
        )
    except Exception as exc:
        print(f"Cache test finale non leggibile, ricalcolo: {exc}")

if use_final_cache:
    df_final_test = pd.read_csv(FINAL_TEST_METRICS_CSV)
    final_test_metrics = final_test_summary.get("per_class", {})
    print("TEST FINALE: dataset filtrati contro i reali del test")
    print("Checkpoint scelto sul validation:", BEST_CHECKPOINT.name)
    print("Metriche finali caricate da cache:", FINAL_TEST_METRICS_PATH)
else:
    final_test_metrics = {}
    print("TEST FINALE: dataset filtrati contro i reali del test")
    print("Checkpoint scelto sul validation:", BEST_CHECKPOINT.name)

    for class_name in FINAL_GENERATE_CLASSES:
        label = FINAL_CLASS_LABELS[class_name]
        generated_dir = FILTERED_DIRS[class_name]
        n_generated = count_pngs(generated_dir)
        if n_generated != N_SELECTED_PER_CLASS:
            raise RuntimeError(
                f"Dataset filtrato incompleto per {class_name}: "
                f"{n_generated} != {N_SELECTED_PER_CLASS}."
            )
        if duplicate_png_groups(generated_dir):
            raise RuntimeError(f"Il dataset filtrato {class_name} contiene duplicati.")

        metrics = evaluate_generated_dir_against_split(
            generated_dir=generated_dir,
            metadata_path=TEST_METADATA_PATH,
            label=label,
            n_images=N_TEST_IMAGES_PER_CLASS,
            seed=EVAL_SEED + label,
        )
        final_test_metrics[class_name] = {
            **metrics,
            "inference_steps": INFERENCE_STEPS,
            "stage": "filtered",
            "class_name": class_name,
            "label": int(label),
            "real_label": label,
            "n_real_reference": N_TEST_IMAGES_PER_CLASS,
            "real_reference_metadata": str(TEST_METADATA_PATH),
            "generated_dir": str(generated_dir),
        }

    n_classes = len(final_test_metrics)
    averages = {
        f"avg_{metric}": round(
            sum(row[metric] for row in final_test_metrics.values()) / n_classes,
            4,
        )
        for metric in final_metric_columns
    }
    final_test_summary = {
        "schema_version": 2,
        "config": final_eval_config,
        "input_signature": final_input_signature,
        "experiment": EXPERIMENT_NAME,
        "best_checkpoint": BEST_CHECKPOINT.name,
        "inference_steps": INFERENCE_STEPS,
        "stage": "filtered",
        "n_generated": N_SELECTED_PER_CLASS,
        "evaluation_reference": "test",
        "classes_evaluated": list(final_test_metrics),
        **averages,
        "per_class": final_test_metrics,
        "timestamp": datetime.now().isoformat(timespec="seconds"),
    }
    with FINAL_TEST_METRICS_PATH.open("w", encoding="utf-8") as handle:
        json.dump(final_test_summary, handle, indent=2, ensure_ascii=False)

    final_rows = [
        {"class": class_name, **metrics}
        for class_name, metrics in final_test_metrics.items()
    ]
    final_rows.append({
        "class": "average",
        **{metric: averages[f"avg_{metric}"] for metric in final_metric_columns},
        "n_generated": sum(row["n_generated"] for row in final_test_metrics.values()),
        "inference_steps": INFERENCE_STEPS,
        "stage": "filtered",
        "class_name": "average",
        "label": "",
        "real_label": "",
        "n_real_reference": N_TEST_IMAGES_PER_CLASS,
        "real_reference_metadata": str(TEST_METADATA_PATH),
        "generated_dir": "",
    })
    df_final_test = pd.DataFrame(final_rows)
    df_final_test.to_csv(FINAL_TEST_METRICS_CSV, index=False)

print("\nMetriche finali sul test:")
print(df_final_test.to_string(index=False))
print("\nSalvate in:", FINAL_TEST_METRICS_PATH, "e", FINAL_TEST_METRICS_CSV)

#### 5.6a Griglie checkpoint e riepilogo finale sul test

Le celle seguenti leggono solo artefatti già prodotti. Le griglie checkpoint vengono generate esclusivamente come pagine numerate; con più di 6 checkpoint non viene prodotta la panoramica completa.

In [ ]:
# Utility per le griglie checkpoint.
CHECKPOINT_PREVIEW_INDEX = 0
MAX_CHECKPOINTS_PER_PAGE = 6
RESULTS_PLOTS_DIR = PLOTS_DIR
RESULTS_METRICS_DIR = METRICS_DIR


def _first_readable_image(directory, preview_index=0):
    directory = Path(directory)
    preferred_names = [
        f"{preview_index:04d}.png",
        f"gen_{preview_index:04d}.png",
    ]
    candidates = [directory / name for name in preferred_names]
    candidates.extend(sorted(directory.glob("*.png")))

    seen = set()
    for path in candidates:
        if path in seen:
            continue
        seen.add(path)
        if not path.exists():
            continue
        try:
            with Image.open(path) as image:
                return path, image.convert("L").copy()
        except Exception as exc:
            warnings.warn(f"Immagine non leggibile, provo fallback: {path} ({exc})")
    return None, None


def _load_checkpoint_preview_records():
    if not Path(EVAL_METRICS_PATH).exists():
        warnings.warn("Metriche checkpoint non trovate: salto la griglia immagini per checkpoint.")
        return []

    with Path(EVAL_METRICS_PATH).open(encoding="utf-8") as handle:
        payload = json.load(handle)

    best_id = Path(BEST_CHECKPOINT).name if "BEST_CHECKPOINT" in globals() else None
    if best_id is None and payload:
        best_id = min(payload, key=lambda row: row.get("avg_FID", math.inf)).get("ckpt_name")

    records = []
    for row in sorted(payload, key=lambda item: int(item.get("step", 0))):
        checkpoint_id = row["ckpt_name"]
        checkpoint_dir = Path(EVAL_DIR) / checkpoint_id
        records.append({
            "checkpoint_id": checkpoint_id,
            "title": checkpoint_id.replace("checkpoint-", "ckpt "),
            "order": int(row.get("step", len(records))),
            "is_best": checkpoint_id == best_id,
            "negative_dir": checkpoint_dir / "negative",
            "positive_dir": checkpoint_dir / "positive",
        })
    return records


def _draw_checkpoint_sample_grid(records, output_path, page_index):
    n_checkpoints = len(records)
    if n_checkpoints == 0:
        return None

    fig_width = max(6.0, 2.2 * n_checkpoints)
    fig, axes = plt.subplots(
        2,
        n_checkpoints,
        figsize=(fig_width, 5.0),
        squeeze=False,
        constrained_layout=True,
    )

    fallbacks = []
    for col, record in enumerate(records):
        for row_index, (class_label, directory_key) in enumerate([
            ("Negativa", "negative_dir"),
            ("Positiva", "positive_dir"),
        ]):
            axis = axes[row_index, col]
            image_path, image = _first_readable_image(
                record[directory_key],
                preview_index=CHECKPOINT_PREVIEW_INDEX,
            )
            if image is None:
                axis.text(0.5, 0.5, "immagine\nmancante", ha="center", va="center", fontsize=8)
                axis.set_facecolor("#f2f2f2")
            else:
                axis.imshow(np.asarray(image), cmap="gray", vmin=0, vmax=255)
                expected = (
                    Path(record[directory_key]) / f"{CHECKPOINT_PREVIEW_INDEX:04d}.png",
                    Path(record[directory_key]) / f"gen_{CHECKPOINT_PREVIEW_INDEX:04d}.png",
                )
                if image_path not in expected:
                    fallbacks.append((record["checkpoint_id"], class_label, image_path.name))

            axis.set_xticks([])
            axis.set_yticks([])
            for spine in axis.spines.values():
                spine.set_visible(bool(record["is_best"]))
                spine.set_edgecolor("crimson")
                spine.set_linewidth(2.0)

            if col == 0:
                axis.set_ylabel(class_label, fontsize=11, fontweight="bold")

        title = record["title"]
        if record["is_best"]:
            title = f"{title}\nBEST"
        axes[0, col].set_title(
            title,
            fontsize=9,
            color="crimson" if record["is_best"] else "black",
            fontweight="bold" if record["is_best"] else "normal",
        )

    fig.suptitle(
        f"Immagini generate per checkpoint - indice {CHECKPOINT_PREVIEW_INDEX} - pagina {page_index}",
        fontsize=13,
    )
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("Salvato:", output_path)

    if fallbacks:
        print("Fallback immagine usati:")
        for checkpoint_id, class_label, filename in fallbacks[:12]:
            print(f"  {checkpoint_id} / {class_label}: {filename}")
        if len(fallbacks) > 12:
            print(f"  ... altri {len(fallbacks) - 12} fallback")
    return output_path

In [ ]:
# Generazione delle sole pagine checkpoint.
def plot_checkpoint_sample_pages():
    records = _load_checkpoint_preview_records()
    if not records:
        return []

    saved_paths = []
    for page, start in enumerate(range(0, len(records), MAX_CHECKPOINTS_PER_PAGE), start=1):
        chunk = records[start:start + MAX_CHECKPOINTS_PER_PAGE]
        page_path = RESULTS_PLOTS_DIR / f"checkpoint_samples_negative_positive_page_{page:02d}.png"
        saved_path = _draw_checkpoint_sample_grid(chunk, page_path, page_index=page)
        if saved_path is not None:
            saved_paths.append(saved_path)
    return saved_paths


checkpoint_plot_paths = plot_checkpoint_sample_pages()

In [ ]:
# Riepilogo finale sul test
def _find_final_test_metrics_csv():
    candidates = []
    if "FINAL_TEST_METRICS_CSV" in globals():
        candidates.append(Path(FINAL_TEST_METRICS_CSV))
    candidates.append(RESULTS_METRICS_DIR / "final_test_metrics.csv")
    for path in candidates:
        if path.exists():
            return path
    return None


def plot_final_test_metrics():
    metrics_path = _find_final_test_metrics_csv()
    if metrics_path is None:
        warnings.warn("Metriche test non trovate: salto il plot finale sul test.")
        return None

    df = pd.read_csv(metrics_path)
    metrics = [metric for metric in ["FID", "IS_mean", "precision", "recall", "density", "coverage"] if metric in df.columns]
    if not metrics:
        warnings.warn(f"Nessuna metrica attesa trovata in {metrics_path}")
        return None

    label_col = "class" if "class" in df.columns else "set_name"
    if label_col not in df.columns:
        label_col = "variant"
        df[label_col] = ["filtered_vs_test"] * len(df)

    plot_df = df[[label_col, *metrics]].copy()
    if "IS_std" in df.columns and "IS_std" not in plot_df.columns:
        plot_df["IS_std"] = df["IS_std"]
    plot_df[label_col] = plot_df[label_col].astype(str)

    output_path = RESULTS_PLOTS_DIR / "final_filtered_vs_test_metrics.png"
    fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)
    axes = axes.ravel()

    if "FID" in metrics:
        plot_df.plot.bar(x=label_col, y="FID", ax=axes[0], color="#4C78A8", legend=False)
        axes[0].set_title("FID sul test - più basso è meglio")
        axes[0].set_xlabel("")
        axes[0].set_ylabel("FID")
        axes[0].grid(axis="y", alpha=0.25)
    else:
        axes[0].axis("off")

    if "IS_mean" in metrics:
        yerr = plot_df["IS_std"] if "IS_std" in df.columns else None
        plot_df.plot.bar(x=label_col, y="IS_mean", yerr=yerr, ax=axes[1], color="#F58518", legend=False, capsize=3)
        axes[1].set_title("Inception Score sul test")
        axes[1].set_xlabel("")
        axes[1].set_ylabel("IS mean")
        axes[1].grid(axis="y", alpha=0.25)
    else:
        axes[1].axis("off")

    prdc_metrics = [metric for metric in ["precision", "recall", "density", "coverage"] if metric in metrics]
    if prdc_metrics:
        x = np.arange(len(plot_df))
        width = 0.8 / max(1, len(prdc_metrics))
        colors = ["#54A24B", "#E45756", "#72B7B2", "#B279A2"]
        for index, metric in enumerate(prdc_metrics):
            axes[2].bar(
                x + (index - (len(prdc_metrics) - 1) / 2) * width,
                plot_df[metric].astype(float),
                width=width,
                label=metric,
                color=colors[index % len(colors)],
            )
        axes[2].set_title("PRDC sul test")
        axes[2].set_xticks(x, plot_df[label_col], rotation=45, ha="right")
        axes[2].set_ylabel("valore")
        axes[2].grid(axis="y", alpha=0.25)
        axes[2].legend(ncol=2, fontsize=8)
    else:
        axes[2].axis("off")

    table_metrics = [metric for metric in ["FID", "IS_mean", "precision", "recall", "density", "coverage"] if metric in metrics]
    table_df = plot_df[[label_col, *table_metrics]].copy()
    for metric in table_metrics:
        table_df[metric] = pd.to_numeric(table_df[metric], errors="coerce").round(4)
    axes[3].axis("off")
    axes[3].set_title("Metriche finali test")
    table = axes[3].table(
        cellText=table_df.values,
        colLabels=table_df.columns,
        loc="center",
        cellLoc="center",
    )
    table.auto_set_font_size(False)
    table.set_fontsize(8)
    table.scale(1.0, 1.25)

    fig.suptitle("Valutazione finale sul test - filtrate a 100 inference step", fontsize=14)
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("Salvato:", output_path)
    print("Metriche lette da:", metrics_path)
    return output_path


final_test_plot_path = plot_final_test_metrics()

## 6. Confronto 02 (fine-tuning completo) vs 04 (LoRA)

Confronta le metriche finali sul test set reale (`diffusers/02_sd21_filtered_100steps` vs `diffusers/04_sd21_lora`, entrambe filtrate, 100 inference step, checkpoint selezionato con lo stesso protocollo di validation) e il costo del training (tempo, energia, CO₂ tracciati con `eco_tracker`, dimensione del checkpoint salvato). Nessuna metrica viene ricalcolata: si leggono solo gli artefatti già salvati da entrambe le pipeline.


In [ ]:
RESULTS_02_DIR = RESULTS_DIR / "diffusers/02_sd21_filtered_100steps"
final_test_02_path = RESULTS_02_DIR / "metrics" / "final_test_metrics.json"
final_test_04_path = FINAL_TEST_METRICS_PATH


def load_final_test_metrics(path, tag):
    if not Path(path).is_file():
        print(f"[{tag}] metriche finali test non disponibili: {path}")
        return None
    with Path(path).open(encoding="utf-8") as handle:
        return json.load(handle)


metrics_02 = load_final_test_metrics(final_test_02_path, "02")
metrics_04 = load_final_test_metrics(final_test_04_path, "04")

if metrics_02 is None or metrics_04 is None:
    print("Confronto qualità 02 vs 04 non disponibile (metriche mancanti su almeno una pipeline).")
else:
    METRIC_ORDER = ["FID", "IS_mean", "precision", "recall", "density", "coverage"]
    LOWER_IS_BETTER = {"FID"}

    def rows_from(payload, tag):
        rows = []
        for class_name, metrics in payload.get("per_class", {}).items():
            if class_name == "average":
                continue
            row = {"pipeline": tag, "class": class_name}
            for metric in METRIC_ORDER:
                row[metric] = metrics.get(metric)
            rows.append(row)
        return rows

    df_compare = pd.DataFrame(rows_from(metrics_02, "02_full_finetune") + rows_from(metrics_04, "04_lora"))
    compare_csv = METRICS_DIR / "comparison_02_vs_04.csv"
    df_compare.to_csv(compare_csv, index=False)
    print("Riepilogo qualità generativa salvato in:", compare_csv)
    print(df_compare.to_string(index=False))

    delta_rows = []
    for class_name in df_compare["class"].unique():
        row_full = df_compare[(df_compare["pipeline"] == "02_full_finetune") & (df_compare["class"] == class_name)]
        row_lora = df_compare[(df_compare["pipeline"] == "04_lora") & (df_compare["class"] == class_name)]
        if row_full.empty or row_lora.empty:
            continue
        entry = {"class": class_name}
        for metric in METRIC_ORDER:
            baseline = row_full.iloc[0][metric]
            candidate = row_lora.iloc[0][metric]
            if pd.isna(baseline) or pd.isna(candidate):
                entry[metric] = None
                continue
            raw_delta_pct = (candidate - baseline) / baseline * 100 if baseline != 0 else float("nan")
            entry[metric] = round(-raw_delta_pct if metric in LOWER_IS_BETTER else raw_delta_pct, 2)
        delta_rows.append(entry)

    df_delta = pd.DataFrame(delta_rows)
    print("\nDelta % (positivo = LoRA migliore del fine-tuning completo):")
    print(df_delta.to_string(index=False))


def count_safetensors_params(path):
    """Conta i parametri di un file .safetensors leggendo solo gli shape, senza caricare i tensori."""
    import math
    from safetensors import safe_open
    total = 0
    with safe_open(str(path), framework="pt") as handle:
        for key in handle.keys():
            shape = handle.get_slice(key).get_shape()
            total += math.prod(shape) if shape else 1
    return total


def sum_completed_sustainability(log_path, label_prefix):
    """Somma le run completate di un log eco_tracker il cui label inizia con label_prefix
    (un training ripreso più volte produce più righe nello stesso file)."""
    log_path = Path(log_path)
    if not log_path.is_file():
        return None
    totals = {"elapsed_seconds": 0.0, "energy_kwh": 0.0, "co2_kg": 0.0, "n_runs": 0}
    with log_path.open(encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            record = json.loads(line)
            if record.get("status") == "completed" and str(record.get("label", "")).startswith(label_prefix):
                totals["elapsed_seconds"] += float(record.get("elapsed_seconds", 0.0))
                totals["energy_kwh"] += float(record.get("energy_kwh", 0.0))
                totals["co2_kg"] += float(record.get("co2_kg", 0.0))
                totals["n_runs"] += 1
    return totals if totals["n_runs"] else None


EXPERIMENT_NAME_02 = "diffusers/02_sd21_filtered_100steps"  # deve combaciare con EXPERIMENT_NAME in 02

lora_cost = sum_completed_sustainability(SUSTAINABILITY_LOG, "finetune_sd21_lora_")
full_ft_cost = sum_completed_sustainability(
    RESULTS_02_DIR / "ecotracker" / "sustainability_finetuning.jsonl", "finetune_sd21_to_"
)

lora_weights_path = (
    Path(BEST_CHECKPOINT) / "pytorch_lora_weights.safetensors"
    if metrics_04 is not None and "BEST_CHECKPOINT" in globals()
    else None
)
lora_size_mb = lora_weights_path.stat().st_size / (1024 ** 2) if lora_weights_path and lora_weights_path.is_file() else None
lora_params = count_safetensors_params(lora_weights_path) if lora_weights_path and lora_weights_path.is_file() else None

full_ft_generation_info = RESULTS_02_DIR / "metrics" / "generation_info.json"
full_unet_size_mb = None
if full_ft_generation_info.is_file():
    with full_ft_generation_info.open(encoding="utf-8") as handle:
        best_checkpoint_02 = json.load(handle).get("best_checkpoint")
    if best_checkpoint_02:
        full_unet_path = (
            PROJECT_ROOT / "experiments" / EXPERIMENT_NAME_02 / "model"
            / best_checkpoint_02 / "unet" / "diffusion_pytorch_model.safetensors"
        )
        if full_unet_path.is_file():
            full_unet_size_mb = full_unet_path.stat().st_size / (1024 ** 2)

print("\n=== Costo ed efficienza del training (LoRA vs fine-tuning completo) ===")
if lora_size_mb is not None:
    extra = f" | {lora_params / 1e6:.2f}M parametri allenati" if lora_params else ""
    print(f"Checkpoint LoRA (best)       : {lora_size_mb:.1f} MB{extra}")
if full_unet_size_mb is not None:
    print(f"Checkpoint U-Net completo    : {full_unet_size_mb:.1f} MB (02, stesso formato safetensors)")
if lora_size_mb is not None and full_unet_size_mb is not None:
    print(f"Riduzione dimensione checkpoint: {(1 - lora_size_mb / full_unet_size_mb) * 100:.1f}%")
if lora_cost is not None:
    print(
        f"Training LoRA            : {lora_cost['elapsed_seconds'] / 3600:.2f} h | "
        f"{lora_cost['energy_kwh']:.4f} kWh | {lora_cost['co2_kg']:.4f} kg CO2 "
        f"({lora_cost['n_runs']} run completate)"
    )
if full_ft_cost is not None:
    print(
        f"Training completo (02)  : {full_ft_cost['elapsed_seconds'] / 3600:.2f} h | "
        f"{full_ft_cost['energy_kwh']:.4f} kWh | {full_ft_cost['co2_kg']:.4f} kg CO2 "
        f"({full_ft_cost['n_runs']} run completate)"
    )
if lora_cost is not None and full_ft_cost is not None and full_ft_cost["elapsed_seconds"] > 0:
    saving_pct = (1 - lora_cost["elapsed_seconds"] / full_ft_cost["elapsed_seconds"]) * 100
    print(f"Risparmio tempo di training: {saving_pct:.1f}%")

efficiency_summary = {
    "lora_checkpoint_size_mb": lora_size_mb,
    "lora_trainable_params": lora_params,
    "full_finetune_checkpoint_size_mb": full_unet_size_mb,
    "lora_training_cost": lora_cost,
    "full_finetune_training_cost": full_ft_cost,
}
efficiency_path = METRICS_DIR / "efficiency_comparison_02_vs_04.json"
with efficiency_path.open("w", encoding="utf-8") as handle:
    json.dump(efficiency_summary, handle, indent=2, ensure_ascii=False)
print("\nRiepilogo efficienza salvato in:", efficiency_path)